In [ ]:
import xarray as xr
import glob
import pandas as pd

# Datasets

## runoff

In [ ]:
grun=xr.open_dataset(r"\\10.230.28.136\cez248024\Objective 1\GRUN\G-RUN_ENSEMBLE_MMM.nc")

In [ ]:
grun

In [ ]:
runoff.to_netcdf(r"\\10.230.28.136\cez248024\Objective 1\(FINAL) Hydroclimate variables used\runoff_1980_2019_0.5degree.nc")

In [ ]:
sr_da = runoff['Runoff']  # This must be a DataArray

## precipitation

In [ ]:
import glob
import os

In [ ]:
import glob
import xarray as xr

pr_files = sorted(glob.glob(r"\\10.230.28.136\cez248024\Preliminary analysis\MSWEP Dataset\0.5 degree\*.nc"))

datasets = [xr.open_dataset(f) for f in pr_files]
pr_ds = xr.concat(datasets, dim="time")

for ds in datasets:
    ds.close()

print(pr_ds)

In [ ]:
all_nan = pr_ds["precipitation"].isnull().all()
print("Are all values NaN?", all_nan.item())

In [ ]:
# Select time range 1980-01-01 to 2019-12-31
pr= pr_ds['precipitation'].sel(time=slice("1980-01-01", "2019-12-31"))

In [ ]:
all_nan = pr.isnull().all()
print("Are all values NaN?", all_nan.item())

In [ ]:
print("Min:", float(pr.min()))
print("Max:", float(pr.max()))
print("Mean:", float(pr.mean()))
print("Std:", float(pr.std()))

In [ ]:
pr1= pr.copy()  # This must be a DataArray

In [ ]:
pr1 = pr1.sortby("lat")

In [ ]:
pr_on_runoff_grid = pr1.interp(
    lat=sr_da.lat,
    lon=sr_da.lon,
    method="nearest"
)

In [ ]:
pr_on_runoff_grid["time"] = pr_on_runoff_grid.indexes["time"].to_period("M").to_timestamp()

In [ ]:
pr_da= pr_on_runoff_grid.copy()

In [ ]:
pr_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\MSWEP Dataset\1980_2019_0.5degree.nc")

In [ ]:
pr_ds = xr.open_dataset(r"\\10.230.28.136\cez248024\Objective 1\MSWEP Dataset\1980_2019_0.5degree.nc")

In [ ]:
pr_da = pr_ds['precipitation']

## soil moisture

In [ ]:
soil_files = sorted(glob.glob(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\gleam_0.5degree\*.nc"))

datasets = [xr.open_dataset(f) for f in soil_files]

sm_ds = xr.concat(datasets, dim="time")

# Close file handles
for ds in datasets:
    ds.close()

In [ ]:
all_nan = sm_ds["SMrz"].isnull().all()
print("Are all values NaN?", all_nan.item())

In [ ]:
# Select time range 1980-01-01 to 2019-12-31
sm= sm_ds['SMrz'].sel(time=slice("1980-01-01", "2019-12-31"))

In [ ]:
print("Min:", float(sm.min()))
print("Max:", float(sm.max()))
print("Mean:", float(sm.mean()))
print("Std:", float(sm.std()))

In [ ]:
sm1= sm.copy()  # This must be a DataArray

In [ ]:
sm1 = sm1.sortby("lat")

In [ ]:
sm_on_runoff_grid = sm1.interp(
    lat=sr_da.lat,
    lon=sr_da.lon,
    method="nearest"
)

In [ ]:
sm_on_runoff_grid["time"] = sm_on_runoff_grid.indexes["time"].to_period("M").to_timestamp()

In [ ]:
# Select time range 1980-01-01 to 2019-12-31
sm_on_runoff_grid= sm_on_runoff_grid.sel(time=slice("1980-01-01", "2019-12-31"))

In [ ]:
sm_da= sm_on_runoff_grid.copy()

In [ ]:
sm_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\1980_2019_0.5degree.nc")

In [ ]:
sm_ds = xr.open_dataset(r"\\10.230.28.136\cez248024\Objective 1\(FINAL) Hydroclimate variables used\smrz(latest)_1980_2019_0.5degree.nc")

In [ ]:
sm_da = sm_ds['SMrz']

# Event Identification

## Wet months 

In [ ]:
# Select baseline period
baseline_pr = pr_da.sel(time=slice("1995-01-01", "2014-12-31"))

In [ ]:
# Compute 90th percentile per grid point
P80 = baseline_pr.quantile(0.80, dim="time")

# P90 dims: (lat, lon)
print(P80.dims)

In [ ]:
# Select baseline period
baseline_sr = sr_da.sel(time=slice("1995-01-01", "2014-12-31"))

In [ ]:
# Compute 90th percentile per grid point
R80 = baseline_sr.quantile(0.80, dim="time")

# P90 dims: (lat, lon)
print(R80.dims)

In [ ]:
pr_aligned, sr_aligned = xr.align(pr_da, sr_da, join="exact")

In [ ]:
wet_months_da_80 = (pr_aligned > P80) & (sr_aligned > R80)

In [ ]:
import numpy as np

In [ ]:
pr_excess = xr.where(wet_months_da, pr_da - P90, np.nan)
sr_excess = xr.where(wet_months_da, sr_da - R90, np.nan)

In [ ]:
pr_excess_80 = xr.where(wet_months_da_80, pr_da - P80, np.nan)
sr_excess_80 = xr.where(wet_months_da_80, sr_da - R80, np.nan)

In [ ]:
pr_excess_85 = xr.where(wet_months_da_85, pr_da - P85, np.nan)
sr_excess_85 = xr.where(wet_months_da_85, sr_da - R85, np.nan)

In [ ]:
I_wet_raw_80 = (pr_excess_80 * sr_excess_80)

In [ ]:
import xarray as xr
import numpy as np

def gringorten_normalize(da, dim="time"):
    """
    Normalize a DataArray using Gringorten plotting position formula
    along a given dimension (default: time).

    P = (r - 0.44) / (N + 0.12)
    """

    # Rank along time (NaNs ignored)
    ranks = da.rank(dim=dim)

    # Count valid values per grid cell
    N = da.count(dim=dim)

    # Gringorten plotting position
    P = (ranks - 0.44) / (N + 0.12)

    return P

In [ ]:
I_wet_80 = gringorten_normalize(I_wet_raw_80)

## dry months

In [ ]:
# Select baseline period
baseline_smrz = sm_da.sel(time=slice("1995-01-01", "2014-12-31"))

In [ ]:
# Compute 10th percentile per grid point
S20 = baseline_smrz.quantile(0.20, dim="time")

# P90 dims: (lat, lon)
print(S20.dims)

In [ ]:
# Compute 10th percentile per grid point
P20 = baseline_pr.quantile(0.20, dim="time")

# P90 dims: (lat, lon)
print(P20.dims)

In [ ]:
dry_months_da_20 = (pr_da< P20) & (sm_da < S20)

In [ ]:
pr_deficit_20 = xr.where(dry_months_da_20, P20 - pr_da, np.nan)
sm_deficit_20 = xr.where(dry_months_da_20, S20 - sm_da, np.nan)

In [ ]:
raw_I_dry_20 = pr_deficit_20*sm_deficit_20

In [ ]:
I_dry_20 = gringorten_normalize(raw_I_dry_20)

## wet and dry phases

In [ ]:
from scipy.ndimage import label
import xarray as xr
import numpy as np

def identify_continuous_events(binary_da, min_len=3):
    """
    Identify continuous events of minimum length in a boolean DataArray.

    Parameters
    ----------
    binary_da : xarray.DataArray
        Boolean DataArray with dims (lat, lon, valid_time)
    min_len : int
        Minimum number of consecutive True timesteps to count as an event

    Returns
    -------
    xarray.DataArray
        Boolean DataArray of same shape as input. True only for timesteps that belong to valid events.
    """
    def process_grid(series):
        """Process 1D boolean array along time axis"""
        labels, num_features = label(series)
        out = np.zeros_like(series, dtype=bool)
        for lbl in range(1, num_features + 1):
            indices = np.where(labels == lbl)[0]
            if len(indices) >= min_len:
                out[indices] = True
        return out

    # Apply along 'valid_time' dimension
    result = xr.apply_ufunc(
        process_grid,
        binary_da,
        input_core_dims=[['time']],
        output_core_dims=[['time']],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[bool]
    )

    return result

In [ ]:
# Wet events: at least 1 month continuous
wet_mask_80 = identify_continuous_events(wet_months_da_80, min_len=1)

In [ ]:
# Dry events: at least 3 months continuous
dry_mask_20 = identify_continuous_events(dry_months_da_20, min_len=3)

### whiplash extraction

In [ ]:
def extract_whiplash_events1(event_A_mask, event_B_mask,
                            intensity_A, intensity_B,
                            max_lag=4):

    events = []

    time = event_A_mask.time.values
    nlats = event_A_mask.lat.size
    nlons = event_A_mask.lon.size
    nt = event_A_mask.time.size

    A  = event_A_mask.transpose("time", "lat", "lon").values
    B  = event_B_mask.transpose("time", "lat", "lon").values
    IA = intensity_A.transpose("time", "lat", "lon").values
    IB = intensity_B.transpose("time", "lat", "lon").values

    for i in range(nlats):
        for j in range(nlons):

            for t in range(nt):

                # A event ends at ta
                if A[t, i, j] and (t == nt-1 or not A[t+1, i, j]):

                    ta = t

                    if IA[ta, i, j] <= 0:
                        continue

                    # find tc = start of A
                    tc = ta
                    while tc > 0 and A[tc-1, i, j]:
                        tc -= 1

                    # search for B event
                    for lag in range(1, max_lag + 1):

                        tb = t + lag
                        if tb >= nt:
                            break

                        # B starts at tb
                        if B[tb, i, j] and (tb == 0 or not B[tb-1, i, j]):

                            if IB[tb, i, j] <= 0:
                                continue

                            # find td = end of B
                            td = tb
                            while td < nt-1 and B[td+1, i, j]:
                                td += 1

                            events.append({
                                "lat_index": i,
                                "lon_index": j,
                                "lat": float(event_A_mask.lat[i]),
                                "lon": float(event_A_mask.lon[j]),

                                "tc_index": int(tc),  # start of A
                                "ta_index": int(ta),  # end of A
                                "tb_index": int(tb),  # start of B
                                "td_index": int(td),  # end of B

                                "tc_time": np.datetime_as_string(time[tc]),
                                "ta_time": np.datetime_as_string(time[ta]),
                                "tb_time": np.datetime_as_string(time[tb]),
                                "td_time": np.datetime_as_string(time[td]),

                                "lag": int(lag),

                                "A_intensity": float(IA[ta, i, j]),
                                "B_intensity": float(IB[tb, i, j])
                            })

                            break

    return events

In [ ]:
dry_to_wet_80 = extract_whiplash_events1(dry_mask_20, wet_mask_80, I_dry_20, I_wet_80, max_lag=6)
print("Number of events:", len(dry_to_wet_80))
dry_to_wet_80[:5]  # see first 5 events

In [ ]:
wet_to_dry_80 = extract_whiplash_events1(wet_mask_80, dry_mask_20, I_wet_80, I_dry_20, max_lag=6)
print("Number of events:", len(wet_to_dry_80))
wet_to_dry_80[:4]  # see first 5 events

In [ ]:
all_whiplash_events_80 = dry_to_wet_80 + wet_to_dry_80

In [ ]:
print("Number of events:", len(all_whiplash_events_80))

In [ ]:
#weight generation for each event
def add_weights_to_events(event_list):
    weighted_events = []

    for ev in event_list:
        lag = ev["lag"]

        # Compute weight
        weight = 1.0 / (lag)

        # Create a new event dictionary INCLUDING weight
        ev_with_weight = ev.copy()
        ev_with_weight["weight"] = weight

        weighted_events.append(ev_with_weight)

    return weighted_events

In [ ]:
weighted_whiplash_events_80 = add_weights_to_events(all_whiplash_events_80)

In [ ]:
weighted_dry_to_wet_80 = add_weights_to_events(dry_to_wet_80)

In [ ]:
weighted_wet_to_dry_80 = add_weights_to_events(wet_to_dry_80)

## dry to wet transitions

In [ ]:
#newest for loop [THIS IS USED!!!]
for event in weighted_dry_to_wet_80:
    # compute score directly from stored intensities
    score = event['A_intensity'] * event['B_intensity'] * event['weight']
    event['score'] = score

In [ ]:
weighted_dry_to_wet_80

In [ ]:
# Set threshold
threshold = 1e-1  # only keep scores above 0.01

filtered_dry_to_wet_80 = [event for event in weighted_dry_to_wet_80 if event['score'] > threshold]

print(f"Number of events after filtering: {len(filtered_dry_to_wet_80)}")

In [ ]:
#for filtered events with score>0.1
#Group events by grid cell
from collections import defaultdict

filtered_events_by_cell_dry_to_wet_80 = defaultdict(list)

for ev in filtered_dry_to_wet_80:
    i = ev["lat_index"]
    j = ev["lon_index"]
    filtered_events_by_cell_dry_to_wet_80[(i, j)].append(ev)

## wet to dry transitions

In [ ]:
#newest for loop [THIS IS USED!!!]
for event in weighted_wet_to_dry_80:
    # compute score directly from stored intensities
    score = event['A_intensity'] * event['B_intensity'] * event['weight']
    event['score'] = score

In [ ]:
# Set threshold
threshold = 1e-1  # only keep scores above 0.01

filtered_wet_to_dry_80 = [event for event in weighted_wet_to_dry_80 if event['score'] > threshold]

print(f"Number of events after filtering: {len(filtered_wet_to_dry_80)}")

In [ ]:
#for filtered events with score>0.1
#Group events by grid cell
from collections import defaultdict

filtered_events_by_cell_wet_to_dry_80 = defaultdict(list)

for ev in filtered_wet_to_dry_80:
    i = ev["lat_index"]
    j = ev["lon_index"]
    filtered_events_by_cell_wet_to_dry_80[(i, j)].append(ev)

In [ ]:
import json
from collections import defaultdict

# Suppose whiplash_dict is your defaultdict(list)
# Convert tuple keys to strings for JSON
json_dict = {str(k): v for k, v in filtered_events_by_cell_wet_to_dry_80.items()}

with open(r"\\10.230.28.136\cez248024\Objective 1\Whiplash results\Cause-Effect\filtered_wet_to_dry_whiplash_events.json", "w") as f:
    json.dump(json_dict, f, indent=4)

## considering all transitions

In [ ]:
all_events_80 = weighted_dry_to_wet_80 + weighted_wet_to_dry_80

In [ ]:
# Set threshold
threshold = 1e-1  # only keep scores above 0.01

filtered_events_80 = [event for event in all_events_80 if event['score'] > threshold]

print(f"Number of events after filtering: {len(filtered_events_80)}")

In [ ]:
#for filtered events with score>0.1
#Group events by grid cell
from collections import defaultdict

filtered_events_by_cell_80 = defaultdict(list)

for ev in filtered_events_80:
    i = ev["lat_index"]
    j = ev["lon_index"]
    filtered_events_by_cell_80[(i, j)].append(ev)

In [ ]:
import json
from collections import defaultdict

# Suppose whiplash_dict is your defaultdict(list)
# Convert tuple keys to strings for JSON
json_dict = {str(k): v for k, v in filtered_events_by_cell_80.items()}

with open(r"\\10.230.28.136\cez248024\Objective 1\Whiplash results\Cause-Effect\filtered_whiplash_events.json", "w") as f:
    json.dump(json_dict, f, indent=4)

In [ ]:
nlat = wet_mask_80.sizes['lat']
nlon = wet_mask_80.sizes['lon']

lat_values = wet_mask_80['lat'].values
lon_values = wet_mask_80['lon'].values

In [ ]:
filtered_max_map_80 = np.full((nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in filtered_events_by_cell_80.items():
    scores = [ev["score"] for ev in ev_list]
    filtered_max_map_80[i, j] = np.max(scores)

In [ ]:
filtered_max_da_80 = xr.DataArray(
    filtered_max_map_80,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="max_filtered_whiplash_score"
)

# whiplash characteristics

## score

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ---------------------------
# GLOBAL FONT SETTINGS
# ---------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14

# ---------------------------
# SETTINGS
# ---------------------------
vmin = 0
vmax = 0.5

ticks = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
tick_labels = ["0.0", "0.1", "0.2", "0.3", "0.4", "≥0.5"]

norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

# ✅ Colorblind-friendly, toned-down colormap
cmap = "viridis"

# ---------------------------
# FIGURE
# ---------------------------
fig = plt.figure(figsize=(10, 8))

ax1 = fig.add_axes([0.08, 0.1, 0.70, 0.8],
                   projection=ccrs.PlateCarree())

cbar_ax = fig.add_axes([0.82, 0.23, 0.02, 0.5])

# ---------------------------
# MAP BASE
# ---------------------------
ax1.set_extent([-180, 180, -90, 90])

ax1.coastlines(resolution='110m', linewidth=0.8)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=0)

# ---------------------------
# MAIN PLOT
# ---------------------------
Main_plot = filtered_max_wholeda_80.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_colorbar=False,
    add_labels=False,
    transform=ccrs.PlateCarree()
)

# ---------------------------
# GRID + TICKS (NO Lat/Lon text)
# ---------------------------
gl = ax1.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)

xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax1.set_xticks(xticks)
ax1.set_yticks(yticks)

ax1.set_xticklabels(
    [f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
    fontsize=14,
    fontweight='bold'
)

ax1.set_yticklabels(
    [f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
    fontsize=14,
    fontweight='bold'
)

ax1.tick_params(
    axis='both',
    direction='in',
    length=6,
    width=1.5
)

# ---------------------------
# BORDER (Cartopy-safe)
# ---------------------------
ax1.spines['geo'].set_linewidth(2)
ax1.spines['geo'].set_edgecolor('black')

# ---------------------------
# COLORBAR
# ---------------------------
cbar = plt.colorbar(
    Main_plot,
    cax=cbar_ax,
    orientation='vertical',
    ticks=ticks
)

cbar.ax.set_yticklabels(tick_labels)

cbar.ax.tick_params(
    labelsize=14,
    width=1.2,
    length=5,
    direction='in'
)

# Make ticks bold
for label in cbar.ax.get_yticklabels():
    label.set_fontweight('bold')

# Colorbar label
cbar.set_label(
    'Whiplash Score',
    fontsize=14,
    fontweight='bold'
)

cbar.outline.set_linewidth(1.5)

# ---------------------------
# TITLE
# ---------------------------
ax1.set_title(
    'Maximum Whiplash Score (1980–2019)',
    fontsize=14,
    fontweight='bold'
)

plt.show()

In [ ]:
filtered_max_wholeda_80.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\int_max_score_1980_2019.nc")

In [ ]:
da = filtered_max_wholeda_80
normalized_score = (da - da.min()) / (da.max() - da.min())

## frequency

In [ ]:
import json
from collections import defaultdict

with open(r"D:\IITD\Doctoral Thesis\Analysis\Global Assessment\filtered_whiplash_events.json", "r") as f:
    loaded_dict = json.load(f)

In [ ]:
whiplash_dict = defaultdict(list, {eval(k): v for k, v in loaded_dict.items()})

In [ ]:
import pandas as pd

rows = []

for (lat_i, lon_i), events in whiplash_dict_c_e.items():
    for e in events:
        rows.append({
            "lat": e["lat"],
            "lon": e["lon"],
            "year": pd.to_datetime(e["ta_time"]).year,
            "score": e["score"]
        })

whiplash_df_80 = pd.DataFrame(rows)

In [ ]:
whiplash_df_80["decade"] = (whiplash_df_80["year"] // 10) * 10

In [ ]:
whiplash_df_80

In [ ]:
grun=xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\GRUN\G-RUN_ENSEMBLE_MMM.nc")

In [ ]:
grun=xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\GRUN\G-RUN_ENSEMBLE_MMM.nc")
# Rename coordinates
grun = grun.rename({"X": "lon", "Y": "lat"})

# Select time range 1980-01-01 to 2019-12-31
runoff= grun.sel(time=slice("1980-01-01", "2019-12-31"))

In [ ]:
nlat = grun.sizes['lat']
nlon = grun.sizes['lon']

lat_values = grun['lat'].values
lon_values = grun['lon'].values

In [ ]:
full_lats = lat_values
full_lons = lon_values

In [ ]:
decadal_freq_80 = (
    whiplash_df_80
    .groupby(["lat", "lon", "decade"])
    .size()
    .reset_index(name="count")
)

In [ ]:
decadal_freq_80

In [ ]:
decadal_freq_80['count'].describe()

In [ ]:
bins = [0, 1, 2, 3, 5, 8]
labels = [
    'Very Low (1)',
    'Low (2)',
    'Moderate (3)',
    'High (4–5)',
    'Very High (≥6)'
]

decadal_freq_80['freq_class'] = pd.cut(
   decadal_freq_80['count'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
class_counts = decadal_freq_80['freq_class'].value_counts().sort_index()

In [ ]:
class_percent = (class_counts / class_counts.sum()) * 100

In [ ]:
class_percent

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams

rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14

plt.figure(figsize=(8,5))
colors = ['#d73027', '#fc8d59', '#fee090', '#91bfdb', '#4575b4']

class_percent.plot(kind='barh', color=colors)

for i, v in enumerate(class_percent):
    if v < 1:
        plt.text(v + 0.2, i, f"{v:.2f}%", va='center', fontsize=12, fontweight='bold')
    else:
        plt.text(v + 0.5, i, f"{v:.1f}%", va='center')

plt.xlabel('Percentage of Grid Cells (%)', fontweight='bold')
plt.xlim(0, 100)
plt.ylabel('Frequency Class (Events/decade)', fontweight='bold')
plt.title('Spatial Distribution of Whiplash Event Frequency', fontweight='bold')

plt.grid(axis='x', linestyle='--', alpha=0.6)

plt.tight_layout()

# ---- Save high resolution ----
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\spatial_distribution_frequency.tif",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
max_decadal_freq_80 = (
    decadal_freq_80
    .groupby(["lat", "lon"])["count"]
    .max()
    .reset_index(name="max_decadal_count")
)

In [ ]:
import xarray as xr
import numpy as np

def df_to_full_grid(df, full_lats, full_lons, value_col="count"):

    grid = xr.DataArray(
        np.full((len(full_lats), len(full_lons)), np.nan),
        coords=[full_lats, full_lons],
        dims=["lat", "lon"]
    )

    for _, r in df.iterrows():
        grid.loc[dict(lat=r["lat"], lon=r["lon"])] = r[value_col]

    return grid

#grid_first = df_to_grid(freq_first)
#grid_last  = df_to_grid(freq_last)
#grid_whole = df_to_grid(max_decadal_freq_80, value_col="max_decadal_count")

In [ ]:
grid_whole = df_to_full_grid(
    max_decadal_freq_80,
    full_lats,
    full_lons,
    value_col="max_decadal_count"
)

In [ ]:
grid_whole.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\int_max_freq_1980_2019.nc")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ---------------------------
# GLOBAL FONT SETTINGS
# ---------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14

# ---------------------------
# FIGURE
# ---------------------------
fig = plt.figure(figsize=(10, 8))

# Make main axis narrower (leave space on right)
ax1 = fig.add_axes([0.08, 0.1, 0.70, 0.8],
                   projection=ccrs.PlateCarree(central_longitude=0))

ax1.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

# ---------------------------
# NORMALIZATION + COLORMAP
# ---------------------------
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=1.5, vmax=3)

# ✅ Colorblind-friendly
cmap = "plasma"

# ---------------------------
# MAIN PLOT
# ---------------------------
Main_plot = grid_whole.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_labels=False,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

# ---------------------------
# COAST + OCEAN
# ---------------------------
ax1.coastlines(resolution='110m', linewidth=1)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=0)

# ---------------------------
# SCATTER OVERLAY
# ---------------------------
lat_vals = grid_whole['lat'].values
lon_vals = grid_whole['lon'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
vals = grid_whole.values

mask = np.isfinite(vals)

ax1.scatter(
    lon2d[mask],
    lat2d[mask],
    c=vals[mask],
    cmap=cmap,
    norm=norm,
    s=1,
    edgecolors='none',
    transform=ccrs.PlateCarree(),
    zorder=5
)

# ---------------------------
# GRIDLINES (NO LAT/LON TEXT)
# ---------------------------
ax1.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)

# ---------------------------
# TICKS
# ---------------------------
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax1.set_xticks(xticks)
ax1.set_yticks(yticks)

ax1.set_xticklabels(
    [f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
    fontsize=14,
    fontweight='bold'
)

ax1.set_yticklabels(
    [f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
    fontsize=14,
    fontweight='bold'
)

ax1.tick_params(
    axis='both',
    direction='in',
    length=6,
    width=1.5
)

# ---------------------------
# BORDER
# ---------------------------
ax1.spines['geo'].set_linewidth(2)
ax1.spines['geo'].set_edgecolor('black')

# ---------------------------
# TITLE
# ---------------------------
ax1.set_title(
    'Maximum Decadal Whiplash Frequency (1980–2019)',
    fontsize=14,
    fontweight='bold'
)

# ---------------------------
# COLORBAR
# ---------------------------
cbar_ax = fig.add_axes([0.82, 0.23, 0.02, 0.5])

vmin = 0
vmax = 3
ticks = np.arange(vmin, vmax + 1, 1)
tick_labels = [str(t) for t in ticks]
tick_labels[-1] = "≥3"

cbar = plt.colorbar(
    Main_plot,
    cax=cbar_ax,
    orientation='vertical',
    ticks=ticks
)

cbar.ax.set_yticklabels(tick_labels)

# Bold + size 14 ticks
for label in cbar.ax.get_yticklabels():
    label.set_fontsize(14)
    label.set_fontweight('bold')

cbar.ax.tick_params(
    width=1.2,
    length=5,
    direction='in'
)

cbar.set_label(
    'Frequency (events/decade)',
    fontsize=14,
    fontweight='bold'
)

cbar.outline.set_linewidth(1.5)

plt.show()

In [ ]:
da1 = grid_whole
normalized_frequency = (da1 - da1.min()) / (da1.max() - da1.min())

In [ ]:
grid_whole

##  duration

In [ ]:
durations = []

for events in whiplash_dict_c_e.values():
    for e in events:
        duration = e['td_index'] - e['tc_index'] + 1
        durations.append(duration)

min(durations), max(durations)

In [ ]:
import numpy as np
durations = np.array(durations)

In [ ]:
import pandas as pd
bins = [3, 4, 7, 10, durations.max()]
labels = ['Short (4)', 'Moderate (5–7)', 'Long (8–10)', 'Very Long (>10)']

duration_class = pd.cut(durations, bins=bins, labels=labels)

class_counts = duration_class.value_counts().sort_index()
class_percent = (class_counts / class_counts.sum()) * 100

In [ ]:
class_percent 

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))

colors = ['#009E73', '#56B4E9', '#E69F00', '#D55E00']

# Labels with percentage
labels = [f"{name}\n{value:.1f}%" for name, value in class_percent.items()]

wedges, texts = plt.pie(
    class_percent,
    labels=labels,
    startangle=90,
    colors=colors,
    labeldistance=1.15
)

# Donut hole
centre_circle = plt.Circle((0,0),0.65,fc='white')
plt.gca().add_artist(centre_circle)

# ✅ Add title inside donut
plt.text(
    0, 0,
    "Distribution of Whiplash\n Event Duration (months)",
    ha='center',
    va='center',
    fontsize=16,
    fontweight='bold'
)

plt.axis('equal')

# ---- Save high resolution ----
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\distribution_duration.tif",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
max_duration = np.full((nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():
    
    durations = []
    
    for ev in ev_list:
        duration = ev["td_index"] - ev["tc_index"] + 1
        durations.append(duration)
    
    if len(durations) > 0:
        max_duration[i, j] = np.max(durations)

In [ ]:
mean_duration = np.full((nlat, nlon), np.nan)

for (i, j), ev_list in filtered_events_by_cell_80.items():
    
    durations = []
    
    for ev in ev_list:
        duration = ev["td_index"] - ev["tc_index"] + 1
        durations.append(duration)
    
    if len(durations) > 0:
        mean_duration[i, j] = np.mean(durations)

In [ ]:
max_duration_da = xr.DataArray(
    max_duration,
    coords={
        "lat": full_lats,
        "lon": full_lons
    },
    dims=["lat", "lon"],
    name="max_whiplash_duration"
)

In [ ]:
max_duration_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\int_max_dur_1980_2019.nc")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ---------------------------
# GLOBAL FONT SETTINGS
# ---------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14

# ---------------------------
# FIGURE (manual layout to avoid overlap)
# ---------------------------
fig = plt.figure(figsize=(10, 8))

ax1 = fig.add_axes([0.08, 0.1, 0.70, 0.8],
                   projection=ccrs.PlateCarree())

ax1.set_extent([-180, 180, -90, 90])

# ---------------------------
# NORMALIZATION + COLORMAP
# ---------------------------
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=5, vmax=10)

# ✅ Colorblind-friendly
cmap = "YlGnBu"

# ---------------------------
# MAIN PLOT
# ---------------------------
Main_plot = max_duration_da.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_labels=False,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

# ---------------------------
# COAST + OCEAN
# ---------------------------
ax1.coastlines(resolution='110m', linewidth=1)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=0)

# ---------------------------
# SCATTER OVERLAY
# ---------------------------
lat_vals = max_duration_da['lat'].values
lon_vals = max_duration_da['lon'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
vals = max_duration_da.values

mask = np.isfinite(vals)

ax1.scatter(
    lon2d[mask],
    lat2d[mask],
    c=vals[mask],
    cmap=cmap,
    norm=norm,
    s=1,
    edgecolors='none',
    transform=ccrs.PlateCarree(),
    zorder=5
)

# ---------------------------
# GRIDLINES
# ---------------------------
ax1.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)

# ---------------------------
# LAT-LON TICKS
# ---------------------------
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax1.set_xticks(xticks)
ax1.set_yticks(yticks)

ax1.set_xticklabels(
    [f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
    fontsize=14,
    fontweight='bold'
)

ax1.set_yticklabels(
    [f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
    fontsize=14,
    fontweight='bold'
)

ax1.tick_params(
    axis='both',
    direction='in',
    length=6,
    width=1.5
)

# ---------------------------
# BORDER
# ---------------------------
ax1.spines['geo'].set_linewidth(2)
ax1.spines['geo'].set_edgecolor('black')

# ---------------------------
# TITLE
# ---------------------------
ax1.set_title(
    'Maximum Whiplash Event Duration (1980–2019)',
    fontsize=14,
    fontweight='bold'
)

# ---------------------------
# COLORBAR (properly spaced)
# ---------------------------
cbar_ax = fig.add_axes([0.82, 0.23, 0.02, 0.5])

vmin = 0
vmax = 10
ticks = np.arange(vmin, vmax + 2, 2)

tick_labels = [str(t) for t in ticks]
tick_labels[-1] = "≥10"

cbar = plt.colorbar(
    Main_plot,
    cax=cbar_ax,
    orientation='vertical',
    ticks=ticks
)

cbar.ax.set_yticklabels(tick_labels)

# ✅ Bold + size 14 ticks
for label in cbar.ax.get_yticklabels():
    label.set_fontsize(14)
    label.set_fontweight('bold')

cbar.ax.tick_params(
    width=1.2,
    length=5,
    direction='in'
)

cbar.set_label(
    'Duration (months)',
    fontsize=14,
    fontweight='bold'
)

cbar.outline.set_linewidth(1.5)

plt.show()

In [ ]:
da2 = max_duration_da
normalized_duration = (da2 - da2.min()) / (da2.max() - da2.min())

# Valdation with GDFC

## GDFC Catalogue

In [ ]:
import os
from datetime import datetime

BASE_DIR = r"\\10.230.28.136\cez248024\Objective 1\GDFC Data\Catalogue"

all_events = []

for event_type in ["Droughts", "Pluvials"]:
    event_type_path = os.path.join(BASE_DIR, event_type)

    for indicator in os.listdir(event_type_path):
        indicator_path = os.path.join(event_type_path, indicator)
        if not os.path.isdir(indicator_path):
            continue

        for continent in os.listdir(indicator_path):
            continent_path = os.path.join(indicator_path, continent)
            if not os.path.isdir(continent_path):
                continue

            for fname in os.listdir(continent_path):
                if not fname.startswith("event."):
                    continue

                fpath = os.path.join(continent_path, fname)

                with open(fpath, "r") as f:
                    lines = f.readlines()

                event = {
                    "event_id": fname,
                    "event_type": event_type.lower(),   # droughts / pluvials
                    "indicator": indicator,             # SMPct / SPI3 / ...
                    "continent": continent,
                    "source": "GDFC"
                }

                for line in lines:
                    parts = line.strip().split()

                    if line.startswith("Event started"):
                        m, y = int(parts[-2]), int(parts[-1])
                        event["start_time"] = datetime(y, m, 1)

                    elif line.startswith("Event stopped"):
                        m, y = int(parts[-2]), int(parts[-1])
                        event["end_time"] = datetime(y, m, 1)

                    elif line.startswith("Latitude Extent"):
                        event["lat_min"] = float(parts[-2])
                        event["lat_max"] = float(parts[-1])

                    elif line.startswith("Longitude Extent"):
                        event["lon_min"] = float(parts[-2])
                        event["lon_max"] = float(parts[-1])

                all_events.append(event)

print(f"Loaded {len(all_events)} GDFC events")

In [ ]:
from collections import defaultdict
import os

folder_counts = defaultdict(lambda: defaultdict(int))

for event_type in ["Droughts", "Pluvials"]:
    event_type_path = os.path.join(BASE_DIR, event_type)

    for indicator in os.listdir(event_type_path):
        indicator_path = os.path.join(event_type_path, indicator)
        if not os.path.isdir(indicator_path):
            continue

        for continent in os.listdir(indicator_path):
            continent_path = os.path.join(indicator_path, continent)
            if not os.path.isdir(continent_path):
                continue

            n_events = sum(
                1 for f in os.listdir(continent_path)
                if f.startswith("event.")
            )

            folder_counts[event_type][continent] += n_events

In [ ]:
for etype, conts in folder_counts.items():
    print(f"\n{etype}")
    for c, n in conts.items():
        print(f"  {c:10s}: {n}")
    print(f"  TOTAL     : {sum(conts.values())}")

In [ ]:
all_events

In [ ]:
import json

with open(r"D:\IITD\Doctoral Thesis\Work and Progress ppts\M1_CEE\global_whiplash\data\gdfc_events.json", "w") as f:
    json.dump(all_events, f, default=str, indent=2)

In [ ]:
import json
from datetime import datetime

with open(r"D:\IITD\Doctoral Thesis\Work and Progress ppts\M1_CEE\global_whiplash\data\gdfc_events.json") as f:
    all_events = json.load(f)

# Convert the time strings back to datetime objects
for e in all_events:
    e["start_time"] = datetime.fromisoformat(e["start_time"])
    e["end_time"] = datetime.fromisoformat(e["end_time"])

print(len(all_events))
print(all_events[0])

In [ ]:
events = sorted(all_events, key=lambda x: x['start_time'])

In [ ]:
from datetime import datetime

START = datetime(1980, 1, 1)
END   = datetime(2016, 12, 31)

gdfc_events = [
    e for e in events
    if e['end_time'] >= START and e['start_time'] <= END
]

In [ ]:
print("Original events:", len(events))
print("Filtered events:", len(gdfc_events))

In [ ]:
def spatial_overlap(e1, e2):
    lat_overlap = not (
        e1['lat_max'] < e2['lat_min'] or
        e2['lat_max'] < e1['lat_min']
    )
    lon_overlap = not (
        e1['lon_max'] < e2['lon_min'] or
        e2['lon_max'] < e1['lon_min']
    )
    return lat_overlap and lon_overlap

In [ ]:
def months_between(t1, t2):
    return (t2.year - t1.year) * 12 + (t2.month - t1.month)

In [ ]:
gdfc_events

In [ ]:
from datetime import datetime
import pandas as pd

In [ ]:
def filter_and_sort_events(events, 
                           start_year=1980, 
                           end_year=2016, 
                           sort_by="ta_time"):
    """
    Filters events to a given year range and sorts them.
    """

    filtered = []

    for e in events:
        ta_time = pd.to_datetime(e["ta_time"])

        if start_year <= ta_time.year <= end_year:
            e_copy = e.copy()
            e_copy["ta_time"] = ta_time
            e_copy["tb_time"] = pd.to_datetime(e["tb_time"])
            filtered.append(e_copy)

    # Sorting options
    if sort_by == "ta_time":
        filtered.sort(key=lambda x: x["ta_time"])
    elif sort_by == "score":
        filtered.sort(key=lambda x: x["score"], reverse=True)
    elif sort_by == "lag":
        filtered.sort(key=lambda x: x["lag"])

    return filtered

In [ ]:
wd_1980_2016 = filter_and_sort_events(
    weighted_wet_to_dry_80, 
    sort_by="ta_time"
)

dw_1980_2016 = filter_and_sort_events(
    weighted_dry_to_wet_80, 
    sort_by="ta_time"
)

## Validation with GDFC

In [ ]:
len(wd_1980_2016)

In [ ]:
gdfc_events

In [ ]:
len(dw_1980_2016)

In [ ]:
nlat

In [ ]:
nlon

In [ ]:
import pandas as pd

time_axis = pd.date_range(
    start="1980-01-01",
    end="2016-12-01",
    freq="MS"   # Month Start
)

ntime = len(time_axis)   # = 360

In [ ]:
from datetime import datetime
import numpy as np

def parse_time(t):
    """Convert numpy/datetime/string to datetime"""
    if isinstance(t, datetime):
        return t
    return np.datetime64(t).astype('datetime64[M]').astype(datetime)

def months_diff(t1, t2):
    """Absolute month difference"""
    return abs((t1.year - t2.year) * 12 + (t1.month - t2.month))

def spatial_match(lat, lon, event):
    return (
        event['lat_min'] <= lat <= event['lat_max'] and
        event['lon_min'] <= lon <= event['lon_max']
    )

def temporal_match(t, event, tol=1):
    # Convert t to datetime.datetime if it's pandas Timestamp or date
    if isinstance(t, pd.Timestamp):
        t = t.to_pydatetime()
    elif isinstance(t, datetime) == False:
        t = datetime.combine(t, datetime.min.time())
    
    # Convert event times to datetime.datetime if needed
    start = event['start_time']
    end = event['end_time']
    if isinstance(start, pd.Timestamp):
        start = start.to_pydatetime()
    if isinstance(end, pd.Timestamp):
        end = end.to_pydatetime()

    # Check tolerance and inclusion
    return (
        abs(months_diff(t, start)) <= tol or
        abs(months_diff(t, end)) <= tol or
        (start <= t <= end)
    )

In [ ]:
def get_time_index(tb_time, time_axis):
    tb = pd.to_datetime(tb_time)
    if tb < time_axis[0] or tb > time_axis[-1]:
        return None
    return (tb.year - 1980) * 12 + (tb.month - 1)

In [ ]:
mask_dry_to_wet = np.zeros((nlat, nlon, ntime), dtype=np.uint8)
mask_wet_to_dry = np.zeros((nlat, nlon, ntime), dtype=np.uint8)

## validation (dry to wet transitions) 

In [ ]:
def validate_whiplash_event(
    lat, lon,
    ta, tb,
    gdfc_events,
    temporal_match,
    spatial_match
):
    """
    Returns True if BOTH:
    - a GDFC drought matches near ta
    - a GDFC pluvial/flood matches near tb
    """

    dry_match = False
    wet_match = False

    for e in gdfc_events:

        # --- Dry side check ---
        if not dry_match and e['event_type'] == 'droughts':
            if spatial_match(lat, lon, e) and temporal_match(ta, e):
                dry_match = True

        # --- Wet side check ---
        elif not wet_match and e['event_type'] in ['pluvials', 'floods']:
            if spatial_match(lat, lon, e) and temporal_match(tb, e):
                wet_match = True

        # Early exit if both found
        if dry_match and wet_match:
            return True

    return False

In [ ]:
for w in dw_1980_2016:

    i = w['lat_index']
    j = w['lon_index']
    lat = w['lat']
    lon = w['lon']

    ta = parse_time(w['ta_time'])   # end of dry
    tb = parse_time(w['tb_time'])   # start of wet

    t_idx = get_time_index(tb, time_axis)
    if t_idx is None:
        continue

    if validate_whiplash_event(
        lat, lon,
        ta, tb,
        gdfc_events,
        temporal_match,
        spatial_match
    ):
        mask_dry_to_wet[i, j, t_idx] = 1

In [ ]:
import numpy as np

# Check if there is at least one True (1) anywhere
if np.any(mask_dry_to_wet):
    print("There are whiplash events detected / validated!")
else:
    print("All zeros: no events detected.")

In [ ]:
# mask_dry_to_wet shape: (nlat, nlon, ntime)
validated_count_dw = np.sum(mask_dry_to_wet, axis=2)

In [ ]:
validated_count_dw.max()

In [ ]:
validated_count_dw.mean()

In [ ]:
np.count_nonzero(validated_count_dw)

In [ ]:
 validated_count_dw = xr.DataArray(
    validated_count_dw,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="validated_count_dw"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = validated_count_dw['lat'].values
lon_vals = validated_count_dw['lon'].values
vals = validated_count_dw.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution='10m', linewidth=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
norm = mcolors.Normalize(vmin=0, vmax=3)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,
    s=40,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.5,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Number of Validated sequences Dry → Wet (1981–2010)',
    fontsize=13,
    fontweight='bold'
)

# ----------------------------------
# 4. Colorbar
# ----------------------------------
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation='vertical',
    fraction=0.035,
    pad=0.04
)

cbar.set_label('Number of Validated sequences', fontsize=12, fontweight='bold')
cbar.set_ticks([0, 1, 2, 3])
cbar.set_ticklabels(['0', '1', '2', '≥3'])
tick_labels[-1] = "≥3"
plt.show()

 Validation Hit Rate (VHR)

In [ ]:
import numpy as np

def build_dw_mask(dw_events, nlat, nlon, time_axis):
    ntime = len(time_axis)
    mask = np.zeros((nlat, nlon, ntime), dtype=np.uint8)

    for e in dw_events:
        i = e['lat_index']
        j = e['lon_index']
        t = np.where(time_axis == e['tb_time'])[0]

        if len(t) == 0:
            continue

        mask[i, j, t[0]] = 1

    return mask

In [ ]:
dw_mask = build_dw_mask(dw_1980_2016, nlat, nlon, time_axis)

In [ ]:
dw_mask

In [ ]:
total_detected_dw = np.sum(dw_mask, axis = 2)

In [ ]:
np.count_nonzero(total_detected_dw)

In [ ]:
# Sum over time axis ONLY
validated_grid_dw = np.sum(mask_dry_to_wet, axis=2)
detected_grid_dw  = np.sum(dw_mask, axis=2)

In [ ]:
VHR_dw = np.where(
    detected_grid_dw > 0,
    100 * validated_grid_dw / detected_grid_dw,
    np.nan
)

In [ ]:
VHR_dw= xr.DataArray(
    VHR_dw,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="VHR_dw"
)

In [ ]:
VHR_dw.max()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = VHR_dw['lat'].values
lon_vals = VHR_dw['lon'].values
vals = VHR_dw.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution='10m', linewidth=1)
#ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
norm = mcolors.Normalize(vmin=0, vmax=100)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,
    s=1,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.0,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Validation Hit Rate (Dry to Wet) (1980–2016)',
    fontsize=13,
    fontweight='bold'
)

# ----------------------------------
# 4. Colorbar
# ----------------------------------
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation='vertical',
    fraction=0.035,
    pad=0.04
)

cbar.set_label('Validation Hit Rate (%)', fontsize=12, fontweight='bold')
cbar.set_ticks([0, 20, 40, 60, 80, 100])
cbar.set_ticklabels(['0', '20', '40', '60', '80', '100'])
#tick_labels[-1] = "≥3"
plt.show()

## validation (wet to dry transitions)

In [ ]:
def validate_wet_to_dry_event(
    lat, lon,
    ta, tb,
    gdfc_events,
    temporal_match,
    spatial_match
):
    """
    Returns True if BOTH:
    - a GDFC pluvial/flood matches near ta (end of wet)
    - a GDFC drought matches near tb (start of dry)
    """

    wet_match = False
    dry_match = False

    for e in gdfc_events:

        # --- Wet side check ---
        if not wet_match and e['event_type'] in ['pluvials', 'floods']:
            if spatial_match(lat, lon, e) and temporal_match(ta, e):
                wet_match = True

        # --- Dry side check ---
        elif not dry_match and e['event_type'] == 'droughts':
            if spatial_match(lat, lon, e) and temporal_match(tb, e):
                dry_match = True

        # Early exit if both found
        if wet_match and dry_match:
            return True

    return False

In [ ]:
for w in wd_1980_2016:

    i = w['lat_index']
    j = w['lon_index']
    lat = w['lat']
    lon = w['lon']

    ta = parse_time(w['ta_time'])   # end of dry
    tb = parse_time(w['tb_time'])   # start of wet

    t_idx = get_time_index(tb, time_axis)
    if t_idx is None:
        continue

    if validate_wet_to_dry_event(
        lat, lon,
        ta, tb,
        gdfc_events,
        temporal_match,
        spatial_match
    ):
        mask_wet_to_dry[i, j, t_idx] = 1

In [ ]:
# mask_dry_to_wet shape: (nlat, nlon, ntime)
validated_count_wd = np.sum(mask_wet_to_dry, axis=2)

In [ ]:
def build_wd_mask(wd_events, nlat, nlon, time_axis):
    """
    Build binary mask for wet → dry whiplash sequences.

    Mask dimensions:
        (lat, lon, time)

    A value of 1 means:
        a validated wet → dry sequence occurs at that grid cell
        at the time of the dry event onset (tb_time).
    """

    ntime = len(time_axis)
    mask = np.zeros((nlat, nlon, ntime), dtype=np.uint8)

    for e in wd_events:
        i = e['lat_index']
        j = e['lon_index']

        # anchor mask at start of second event (dry onset)
        t = np.where(time_axis == e['tb_time'])[0]

        if len(t) == 0:
            continue

        mask[i, j, t[0]] = 1

    return mask

In [ ]:
wd_mask = build_wd_mask(wd_1980_2016, nlat, nlon, time_axis)

In [ ]:
validated_grid_wd = np.sum(mask_wet_to_dry, axis=2)
detected_grid_wd  = np.sum(wd_mask, axis=2)

In [ ]:
VHR_wd = np.where(
    detected_grid_wd > 0,
    100 * validated_grid_wd / detected_grid_wd,
    np.nan
)

In [ ]:
VHR_wd= xr.DataArray(
    VHR_wd,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="VHR_wd"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = VHR_wd['lat'].values
lon_vals = VHR_wd['lon'].values
vals = VHR_wd.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution='10m', linewidth=1)
#ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
norm = mcolors.Normalize(vmin=0, vmax=100)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,
    s=1,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.0,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Validation Hit Rate (Wet to Dry) (1980–2016)',
    fontsize=13,
    fontweight='bold'
)

# ----------------------------------
# 4. Colorbar
# ----------------------------------
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation='vertical',
    fraction=0.035,
    pad=0.04
)

cbar.set_label('Validation Hit Rate (%)', fontsize=12, fontweight='bold')
cbar.set_ticks([0, 20, 40, 60, 80, 100])
cbar.set_ticklabels(['0', '20', '40', '60', '80', '100'])
#tick_labels[-1] = "≥3"
plt.show()

## combining both

In [ ]:
validated_grid = validated_grid_dw + validated_grid_wd
detected_grid = detected_grid_dw + detected_grid_wd

In [ ]:
VHR = np.where(
    detected_grid > 0,
    100 * validated_grid / detected_grid,
    np.nan
)

In [ ]:
VHR= xr.DataArray(
    VHR,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="VHR"
)

In [ ]:
VHR.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\VHR_c_e.nc")

In [ ]:
VHR.mean()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = VHR['lat'].values
lon_vals = VHR['lon'].values
vals = VHR.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

# Make border thicker
for spine in ax.spines.values():
    spine.set_linewidth(2)   # increase thickness (try 1.5–3)
    spine.set_color('black')
    
ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution='10m', linewidth=1)
#ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
norm = mcolors.Normalize(vmin=0, vmax=100)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,
    s=1,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Validation Hit Rate [Cause-Effect] (1980–2016)',
    fontsize=13,
    fontweight='bold',
    fontname='Times New Roman'
)

# ----------------------------------
# 4. Colorbar
# ----------------------------------
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation='vertical',
    fraction=0.025,   # smaller width (default ~0.035)
    pad=0.03,
    shrink=0.8,       # reduces height
    aspect=25         # larger = thinner bar
)


cbar.set_label('Validation Hit Rate (%)', fontsize=12, fontweight='bold')
cbar.set_ticks([0, 20, 40, 60, 80, 100])
cbar.set_ticklabels(['0', '20', '40', '60', '80', '100'])
#tick_labels[-1] = "≥3"
plt.show()

In [ ]:
FCR_dw = np.where(
    detected_grid_dw > 0,
    100*(1 - ((validated_grid_dw) / (detected_grid_dw))),
    np.nan
)

In [ ]:
FCR_dw= xr.DataArray(
    FCR_dw,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="FCR_dw"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = FCR_dw['lat'].values
lon_vals = FCR_dw['lon'].values
vals = FCR_dw.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution='10m', linewidth=1)
#ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
norm = mcolors.Normalize(vmin=0, vmax=100)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,
    s=40,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.5,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Unvalidated Detection Rate (%) (Dry to Wet) (1981–2010)',
    fontsize=13,
    fontweight='bold'
)

# ----------------------------------
# 4. Colorbar
# ----------------------------------
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation='vertical',
    fraction=0.035,
    pad=0.04
)

cbar.set_label('Unvalidated Detection Rate (%)', fontsize=12, fontweight='bold')
cbar.set_ticks([0, 20, 40, 60, 80, 100])
cbar.set_ticklabels(['0', '20', '40', '60', '80', '100'])
#tick_labels[-1] = "≥3"
plt.show()

In [ ]:
FCR_wd = np.where(
    detected_grid_wd > 0,
    100*(1 - ((validated_grid_wd) / (detected_grid_wd))),
    np.nan
)

In [ ]:
FCR_wd= xr.DataArray(
    FCR_wd,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="FCR_wd"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = FCR_dw['lat'].values
lon_vals = FCR_dw['lon'].values
vals = FCR_dw.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(2)   # increase thickness (try 1.5–3)
    spine.set_color('black')
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution='10m', linewidth=1)
#ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
norm = mcolors.Normalize(vmin=0, vmax=100)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,
    s=40,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.5,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Unvalidated Detection Rate (%) (Wet to Dry) (1981–2010)',
    fontsize=13,
    fontweight='bold'
)

# ----------------------------------
# 4. Colorbar
# ----------------------------------
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation='vertical',
    fraction=0.035,
    pad=0.04
)

cbar.set_label('Unvalidated Detection Rate (%)', fontsize=12, fontweight='bold')
cbar.set_ticks([0, 20, 40, 60, 80, 100])
cbar.set_ticklabels(['0', '20', '40', '60', '80', '100'])
#tick_labels[-1] = "≥3"
plt.show()

In [ ]:
FCR = np.where(
    detected_grid > 0,
    100 * (1 - (validated_grid / detected_grid)),
    np.nan
)

In [ ]:
FCR= xr.DataArray(
    FCR,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="FCR"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = FCR['lat'].values
lon_vals = FCR['lon'].values
vals = FCR.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(2)   # increase thickness (try 1.5–3)
    spine.set_color('black')
    
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

ax.set_extent([-180, 180, -90, 90])
for spine in ax.spines.values():
    spine.set_linewidth(2)   # increase thickness (try 1.5–3)
    spine.set_color('black')

ax.coastlines(resolution='10m', linewidth=1)
#ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
norm = mcolors.Normalize(vmin=0, vmax=100)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,
    s=1,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.0,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Unvalidated Detection Rate [Cause-Effect] (1980–2016)',
    fontsize=13,
    fontweight='bold',
    fontname='Times New Roman'
)

# ----------------------------------
# 4. Colorbar
# ----------------------------------
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation='vertical',
    fraction=0.035,
    pad=0.04
)

cbar.set_label('Unvalidated Detection Rate (%)', fontsize=12, fontweight='bold')
cbar.set_ticks([0, 20, 40, 60, 80, 100])
cbar.set_ticklabels(['0', '20', '40', '60', '80', '100'])
#tick_labels[-1] = "≥3"
plt.show()

## Overlap with flood/drought regions

In [ ]:
import numpy as np

# Initialize classification grid
class_grid_dw = np.zeros_like(detected_grid_dw, dtype=int)

# Class 1: detected but not validated
class_grid_dw[(detected_grid_dw > 0) & (validated_grid_dw == 0)] = 1

# Class 2: detected and validated
class_grid_dw[(validated_grid_dw > 0)] = 2

In [ ]:
class_grid_dw

In [ ]:
overlap_dw= xr.DataArray(
    class_grid_dw,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="overlap_dw"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = overlap_dw['lat'].values
lon_vals = overlap_dw['lon'].values
vals = overlap_dw.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution='10m', linewidth=1)
#ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
#norm = mcolors.Normalize(vmin=0, vmax=100)
# Discrete colormap
cmap = mcolors.ListedColormap(['orange', 'red'])
norm = mcolors.BoundaryNorm([0.5, 1.5, 2.5], cmap.N)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap=cmap,
    norm=norm,
    s=18,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.5,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Detected vs Validated Whiplash Sequences (Dry to Wet) (1981–2010)',
    fontsize=13,
    fontweight='bold'
)

# ----------------------------------
# 4. Colorbar
# ----------------------------------
#cbar = plt.colorbar(
    #sc,
    #ax=ax,
    #orientation='vertical',
    #fraction=0.035,
    #pad=0.04
#)

#cbar.set_label('False Construction Rate (%)', fontsize=12, fontweight='bold')
#cbar.set_ticks([0, 20, 40, 60, 80, 100])
#cbar.set_ticklabels(['0', '20', '40', '60', '80', '100'])
#tick_labels[-1] = "≥3"
#plt.show()

from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           label='Detected only',
           markerfacecolor='orange', markersize=8),

    Line2D([0], [0], marker='o', color='w',
           label='Detected + Validated (GDFC)',
           markerfacecolor='red', markersize=8)
]

ax.legend(
    handles=legend_elements,
    loc='lower left',
    frameon=True,
    fontsize=11
)

plt.show()


In [ ]:
import numpy as np

# Initialize classification grid
class_grid_wd = np.zeros_like(detected_grid_wd, dtype=int)

# Class 1: detected but not validated
class_grid_wd[(detected_grid_wd > 0) & (validated_grid_wd == 0)] = 1

# Class 2: detected and validated
class_grid_wd[(validated_grid_wd > 0)] = 2

In [ ]:
overlap_wd= xr.DataArray(
    class_grid_wd,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="overlap_wd"
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = overlap_wd['lat'].values
lon_vals = overlap_wd['lon'].values
vals = overlap_wd.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution='10m', linewidth=1)
ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
#norm = mcolors.Normalize(vmin=0, vmax=100)
# Discrete colormap
cmap = mcolors.ListedColormap(['orange', 'red'])
norm = mcolors.BoundaryNorm([0.5, 1.5, 2.5], cmap.N)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap=cmap,
    norm=norm,
    s=18,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.5,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Detected vs Validated Whiplash Sequences (Wet to Dry) (1981–2010)',
    fontsize=13,
    fontweight='bold'
)

# ----------------------------------
# 4. Colorbar
# ----------------------------------
#cbar = plt.colorbar(
    #sc,
    #ax=ax,
    #orientation='vertical',
    #fraction=0.035,
    #pad=0.04
#)

#cbar.set_label('False Construction Rate (%)', fontsize=12, fontweight='bold')
#cbar.set_ticks([0, 20, 40, 60, 80, 100])
#cbar.set_ticklabels(['0', '20', '40', '60', '80', '100'])
#tick_labels[-1] = "≥3"
#plt.show()

from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           label='Detected only',
           markerfacecolor='orange', markersize=8),

    Line2D([0], [0], marker='o', color='w',
           label='Detected + Validated (GDFC)',
           markerfacecolor='red', markersize=8)
]

ax.legend(
    handles=legend_elements,
    loc='lower left',
    frameon=True,
    fontsize=11
)

plt.show()


In [ ]:
import numpy as np

# Initialize classification grid
class_grid = np.zeros_like(detected_grid, dtype=int)

# Class 1: detected but not validated
class_grid[(detected_grid> 0) & (validated_grid == 0)] = 1

# Class 2: detected and validated
class_grid[(validated_grid> 0)] = 2

In [ ]:
overlap= xr.DataArray(
    class_grid,
    dims=["lat", "lon"],
    coords={"lat": lat_values, "lon": lon_values},
    name="overlap"
)

In [ ]:
overlap.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\det_val_c_e.nc")

In [ ]:
detected_grid

In [ ]:
import xarray as xr

In [ ]:
det_val_c_e = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\detected_validated_c_e.nc")

In [ ]:
overlap =  det_val_c_e['overlap']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ----------------------------------
# 1. Extract valid POD points
# ----------------------------------
lat_vals = overlap['lat'].values
lon_vals = overlap['lon'].values
vals = overlap.values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

mask = np.isfinite(vals) & (vals > 0)   # keep only meaningful PODs

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

print("Number of plotted points:", len(val_pts))

# ----------------------------------
# 2. Figure & map
# ----------------------------------
fig, ax = plt.subplots(
    figsize=(10, 8),
    subplot_kw={'projection': ccrs.PlateCarree()}
)

# Make border thicker
for spine in ax.spines.values():
    spine.set_linewidth(2)   # increase thickness (try 1.5–3)
    spine.set_color('black')
ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution='10m', linewidth=1)
#ax.add_feature(cfeature.BORDERS, linewidth=0.6)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey')

# ----------------------------------
# 3. Scatter plot ONLY
# ----------------------------------
#norm = mcolors.Normalize(vmin=0, vmax=100)
# Discrete colormap
cmap = mcolors.ListedColormap(['orange', 'red'])
norm = mcolors.BoundaryNorm([0.5, 1.5, 2.5], cmap.N)

sc = ax.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap=cmap,
    norm=norm,
    s=1,                # <-- BIGGER SIZE
    marker='o',
    linewidths=0.0,
    edgecolor = 'black',# <-- NO EDGES
    transform=ccrs.PlateCarree(),
    zorder=5
)

ax.set_title(
    'Detected vs Validated Whiplash Sequences [Integrated] (1980–2016)',
    fontname='Times New Roman', 
    fontsize=13,
    fontweight='bold'
)
# ----------------------------------
# Map styling (CORRECTED)
# ----------------------------------
ax.set_extent([-180, 180, -90, 90])

ax.coastlines(resolution="110m", linewidth=0.8)
ax.add_feature(cfeature.OCEAN, facecolor="lightgrey", edgecolor="none")

# Gridlines (no auto labels)
gl = ax.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)

# Set ticks
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax.set_xticks(xticks)
ax.set_yticks(yticks)

# Custom tick labels (NO "Lat/Lon", only degrees)
ax.set_xticklabels(
    [f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
    fontname='Times New Roman',
    fontsize=14,
    fontweight='bold'
)

ax.set_yticklabels(
    [f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
    fontname='Times New Roman',
    fontsize=14,
    fontweight='bold'
)

# Tick appearance
ax.tick_params(
    axis='both',
    which='major',
    direction='in',
    length=6,
    width=1.5
)

# Thick map boundary (IMPORTANT for Cartopy)
ax.spines['geo'].set_linewidth(2)
ax.spines['geo'].set_edgecolor('black')
# ----------------------------------
# 4. Colorbar
# ----------------------------------
#cbar = plt.colorbar(
    #sc,
    #ax=ax,
    #orientation='vertical',
    #fraction=0.035,
    #pad=0.04
#)

#cbar.set_label('False Construction Rate (%)', fontsize=12, fontweight='bold')
#cbar.set_ticks([0, 20, 40, 60, 80, 100])
#cbar.set_ticklabels(['0', '20', '40', '60', '80', '100'])
#tick_labels[-1] = "≥3"
#plt.show()

from matplotlib.lines import Line2D

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           label='Detected only',
           markerfacecolor='orange', markersize=8),

    Line2D([0], [0], marker='o', color='w',
           label='Detected + Validated (GDFC)',
           markerfacecolor='red', markersize=8)
]
legend = ax.legend(
    handles=legend_elements,
    loc='lower left',
    frameon=True,
    fontsize=14
)

# Set font properties properly
for text in legend.get_texts():
    text.set_fontname('Times New Roman')
    text.set_fontweight('bold')
    text.set_fontsize(14)

plt.show()


## overall percentage of validated sequences all around the world. 

In [ ]:
total_detected_dw = np.sum(detected_grid_dw)
total_validated_dw = np.sum(validated_grid_dw)

VHR_dw = 100 * total_validated_dw / total_detected_dw

In [ ]:
VHR_dw

In [ ]:
total_detected_wd = np.sum(detected_grid_wd)
total_validated_wd = np.sum(validated_grid_wd)

VHR_wd = 100 * total_validated_wd / total_detected_wd

In [ ]:
VHR_wd

In [ ]:
total_detected_all = total_detected_dw + total_detected_wd
total_validated_all = total_validated_dw + total_validated_wd

VHR_all = 100 * total_validated_all / total_detected_all

In [ ]:
VHR_all

In [ ]:
total_detected_all

In [ ]:
total_validated_all

# Gridded water stress linkage analysis

## ET/PET Dataset

In [ ]:
e_files = sorted(glob.glob(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\Water Stress\E_0.5degree\*.nc"))

datasets = [xr.open_dataset(f) for f in e_files]

e_ds = xr.concat(datasets, dim="time")

# Close file handles
for ds in datasets:
    ds.close()

In [ ]:
e_on_runoff_grid = e1.interp(
    lat=sr_da.lat,
    lon=sr_da.lon,
    method="nearest"
)

In [ ]:
e_on_runoff_grid["time"] = e_on_runoff_grid.indexes["time"].to_period("M").to_timestamp()

In [ ]:
# Select time range 1980-01-01 to 2019-12-31
e_on_runoff_grid= e_on_runoff_grid.sel(time=slice("1980-01-01", "2019-12-31"))

In [ ]:
e_da= e_on_runoff_grid.copy()

In [ ]:
e_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\Water Stress\ e_1980_2019_0.5degree.nc")

In [ ]:
ep_files = sorted(glob.glob(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\Water Stress\Ep_0.5degree\*.nc"))

datasets = [xr.open_dataset(f) for f in ep_files]

ep_ds = xr.concat(datasets, dim="time")

# Close file handles
for ds in datasets:
    ds.close()

In [ ]:
ep1= ep.copy()  # This must be a DataArray

In [ ]:
ep1 = ep1.sortby("lat")

In [ ]:
ep_on_runoff_grid = ep1.interp(
    lat=sr_da.lat,
    lon=sr_da.lon,
    method="nearest"
)

In [ ]:
ep_on_runoff_grid["time"] = ep_on_runoff_grid.indexes["time"].to_period("M").to_timestamp()

In [ ]:
# Select time range 1980-01-01 to 2019-12-31
ep_on_runoff_grid= ep_on_runoff_grid.sel(time=slice("1980-01-01", "2019-12-31"))

In [ ]:
ep_da= ep_on_runoff_grid.copy()

In [ ]:
ep_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\Water Stress\ ep_1980_2019_0.5degree.nc")

In [ ]:
e_da = e_da.where(e_da >= 0)

In [ ]:
ep_da = ep_da.where(ep_da>=0)
ep_da = ep_da.where(ep_da > 1e-3)  # threshold matters

In [ ]:
water_stress_da = xr.where(
    (ep_da > 0) & (e_da > 0),
    e_da / ep_da,
    np.nan
)

In [ ]:
water_stress_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\Water Stress\water_stress_1980_2019.nc")

## ECA (Trigger rate)

In [ ]:
water_stress_ds =xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\GLEAM v4\Water Stress\water_stress_1980_2019.nc")

In [ ]:
water_stress_da = water_stress_ds["__xarray_dataarray_variable__"]

In [ ]:
# Select baseline period
baseline_stress = water_stress_da.sel(time=slice("1995-01-01", "2014-12-31"))

In [ ]:
# Compute 90th percentile per grid point
ST20 = baseline_stress.quantile(0.20, dim="time")

# P90 dims: (lat, lon)
print(ST20.dims)

In [ ]:
stress_mask = water_stress_da < ST20

In [ ]:
times = pd.date_range("1980-01-01", "2019-12-01", freq="MS")  # 480 months
nT = len(times)  # 480

In [ ]:
whiplash_score_map = np.full((nT, nlat, nlon), np.nan)

In [ ]:
filtered_events_by_cell_80

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():

    for ev in ev_list:

        # full whiplash sequence
        t_start = ev["tc_index"]
        t_end   = ev["td_index"] + 1   # +1 because Python range excludes last index

        for t_idx in range(t_start, t_end):

            if np.isnan(whiplash_score_map[t_idx, i, j]):
                whiplash_score_map[t_idx, i, j] = ev["score"]
            else:
                whiplash_score_map[t_idx, i, j] = max(
                    whiplash_score_map[t_idx, i, j],
                    ev["score"]
                )

In [ ]:
whiplash_score_map

In [ ]:
whiplash_score_da = xr.DataArray(
    data=whiplash_score_map,          # numpy array with all values filled
    coords={
        'time': times,                # e.g., np.arange(2000, 2015)
        'latitude': lat_values,
        'longitude': lon_values
    },
    dims=('time', 'latitude', 'longitude'),
    name='whiplash_score'
)

In [ ]:
def ECA_trigger_grid(X, Y, deltaT=5, tau=0):
    """
    X : 1D numpy array (binary whiplash indicator, 1 during whiplash sequence)
    Y : 1D numpy array (binary water stress indicator, 1 during stress)
    deltaT : maximum allowed delay after whiplash sequence ends (months)
    tau : minimum lag after whiplash sequence ends (default 0)

    Returns:
    r_trig : trigger coincidence rate
    """

    import numpy as np

    # indices where whiplash exists
    tX = np.where(X == 1)[0]
    tY = np.where(Y == 1)[0]

    NX = len(tX)
    NY = len(tY)

    if NX == 0 or NY == 0:
        return np.nan

    # Identify contiguous whiplash sequences
    sequences = []
    start = tX[0]

    for i in range(1, NX):
        if tX[i] != tX[i-1] + 1:
            sequences.append((start, tX[i-1]))
            start = tX[i]

    sequences.append((start, tX[-1]))

    # Check stress AFTER sequence end
    triggered = 0
    total_sequences = len(sequences)

    for seq in sequences:

        tc, td = seq   # td = end of whiplash sequence

        # define allowed window AFTER sequence
        start_window = td + tau
        end_window   = td + tau + deltaT

        # ensure within bounds
        start_window = max(start_window, 0)
        end_window   = min(end_window, len(Y)-1)

        if start_window <= end_window:

            if np.any(Y[start_window:end_window+1] == 1):
                triggered += 1

    r_trig = triggered / total_sequences

    return r_trig

In [ ]:
whiplash_month = (whiplash_score_da > 0).astype(int)

In [ ]:
stress_month = (water_stress_da < ST20).astype(int)

lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

trigger_rate_map = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_map[i, j] = ECA_trigger_grid(X, Y)

In [ ]:
lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

trigger_rate_map_5 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_map_5[i, j] = ECA_trigger_grid(X, Y, deltaT=5, tau=0)

In [ ]:
lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

trigger_rate_map_3 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_map_3[i, j] = ECA_trigger_grid(X, Y, deltaT=3, tau=0)

In [ ]:
lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

trigger_rate_map_1 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        # extract 1D time series for this grid cell
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        # compute trigger coincidence rate
        trigger_rate_map_1[i, j] = ECA_trigger_grid(X, Y, deltaT=1, tau=0)

In [ ]:
# convert to xarray DataArray
trigger_rate_da_5 = xr.DataArray(
    data=trigger_rate_map_5,
    coords={'latitude': whiplash_month.latitude, 'longitude': whiplash_month.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
# convert to xarray DataArray
trigger_rate_da_3 = xr.DataArray(
    data=trigger_rate_map_3,
    coords={'latitude': whiplash_month.latitude, 'longitude': whiplash_month.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
trigger_rate_da_3.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_all_1980_2019_3.nc")

In [ ]:
# convert to xarray DataArray
trigger_rate_da_1 = xr.DataArray(
    data=trigger_rate_map_1,
    coords={'latitude': whiplash_month.latitude, 'longitude': whiplash_month.longitude},
    dims=('latitude', 'longitude'),
    name='trigger_coincidence_rate',
    attrs={'long_name': 'Trigger coincidence rate between whiplash and water stress',
           'units': 'fraction (0-1)'}
)

In [ ]:
all_nan = trigger_rate_da.isnull().all()
print("Are all values NaN?", all_nan.item())

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from matplotlib.ticker import MultipleLocator, AutoMinorLocator

fig, ax1 = plt.subplots(
    1, 1,
    subplot_kw={'projection': ccrs.PlateCarree(central_longitude=0)},
    figsize=(10, 8),
    constrained_layout=False
)

[x.set_linewidth(1.8) for x in ax1.spines.values()]    
ax1.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

# normalizer (you can change vmin/vmax)
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter = 0.5, vmax=1)

Main_plot = trigger_rate_da_5.plot(
    ax=ax1,
    cmap='turbo',
    norm=norm,
    add_labels=False,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax1.coastlines(resolution='10m', linewidth=1)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', edgecolor='black', linewidth=0.6, zorder=2)
ax1.add_feature(cfeature.BORDERS, linewidth=0.6)
ax1.set_title('Trigger Coincidence Rate between Whiplash & Water Stress', fontsize=13, pad=8, fontname='Times New Roman', weight='bold')
fig.subplots_adjust(top=0.9, right=0.8)

lat_vals = trigger_rate_da_5['latitude'].values
lon_vals = trigger_rate_da_5['longitude'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
vals = trigger_rate_da_5.values

mask = np.isfinite(vals)

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

sc = ax1.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,                 # SAME norm as raster
    s=1,                      # <-- increase this (try 20–60)
    marker='o',
    edgecolors='black',
    linewidths=0.0,
    transform=ccrs.PlateCarree(),
    zorder=5
)

# Colorbar
cbar_ax = fig.add_axes([0.82, 0.23, 0.02, 0.5])
vmin = 0
vmax = 1  # e.g., 1.21

tick_interval = 0.1
ticks = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]   # last tick slightly below vmax
tick_labels = ["0.0", "0.2", "0.4", "0.6", "0.8", "1.0"]
#tick_labels[-1] = "≥0.6"   # replace last tick label

# Create colorbar FIRST
cbar = plt.colorbar(
    Main_plot,
    cax=cbar_ax,
    orientation='vertical',
    ticks=ticks
)

# Set tick labels
cbar.ax.set_yticklabels(tick_labels)

# THEN add manual extension label
cbar.ax.text(
    0.5, 1.02, "1",
    ha="center",
    va="bottom",
    transform=cbar.ax.transAxes,
    fontsize=11.7,
    fontweight="bold"
)

#cbar.ax1.set_yticklabels(tick_labels)
cbar.outline.set_linewidth(1.2)
cbar.ax.tick_params(which='both', labelsize=11.7, width=1, length=5, direction='in')
cbar.ax.set_ylabel('Trigger coincidence rate', labelpad=1, fontsize=13, fontweight='bold')

In [ ]:
import numpy as np

def trigger_rate_significance(X, Y, tau=0, delT=5, n_iter=1000, seed=None):
    """
    X, Y : 1D binary arrays
    tau : lag
    delT : window length
    n_iter : number of Monte Carlo simulations
    Returns: observed rate, p-value
    """
    np.random.seed(seed)
    
    # observed trigger rate
    r_obs = ECA_trigger_grid(X, Y, deltaT=5, tau=0)
    
    # number of events
    NX = X.sum()
    NY = Y.sum()
    
    if NX == 0 or NY == 0:
        return r_obs, np.nan
    
    r_null = []
    T = len(X)
    
    for _ in range(n_iter):
        # randomly place X and Y events in time
        X_rand = np.zeros(T, dtype=int)
        Y_rand = np.zeros(T, dtype=int)
        
        X_idx = np.random.choice(T, NX, replace=False)
        Y_idx = np.random.choice(T, NY, replace=False)
        
        X_rand[X_idx] = 1
        Y_rand[Y_idx] = 1
        
        r_null.append(ECA_trigger_grid(X_rand, Y_rand, tau))
    
    r_null = np.array(r_null)
    
    # p-value: fraction of random trigger rates >= observed
    p_value = (r_null >= r_obs).sum() / n_iter
    
    return r_obs, p_value

In [ ]:
lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

p_value_map = np.full((lat_size, lon_size), np.nan)
trigger_rate_map = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        r_obs, p_val = trigger_rate_significance(X, Y, tau=0, delT=5, n_iter=500)
        
        trigger_rate_map[i, j] = r_obs
        p_value_map[i, j] = p_val

In [ ]:
lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

p_value_map_1 = np.full((lat_size, lon_size), np.nan)
trigger_rate_map_1 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        r_obs_1, p_val_1 = trigger_rate_significance(X, Y, tau=0, delT=1, n_iter=500)
        
        trigger_rate_map_1[i, j] = r_obs_1
        p_value_map_1[i, j] = p_val_1

In [ ]:
lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

p_value_map_3 = np.full((lat_size, lon_size), np.nan)
trigger_rate_map_3 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        r_obs_3, p_val_3 = trigger_rate_significance(X, Y, tau=0, delT=3, n_iter=500)
        
        trigger_rate_map_3[i, j] = r_obs_3
        p_value_map_3[i, j] = p_val_3

In [ ]:
lat_size = whiplash_month.latitude.size
lon_size = whiplash_month.longitude.size

p_value_map1 = np.full((lat_size, lon_size), np.nan)
trigger_rate_map1 = np.full((lat_size, lon_size), np.nan)

for i in range(lat_size):
    for j in range(lon_size):
        X = whiplash_month[:, i, j].values
        Y = stress_month[:, i, j].values
        
        r_obs, p_val = trigger_rate_significance(X, Y, tau=1, delT=5, n_iter=500)
        
        trigger_rate_map1[i, j] = r_obs
        p_value_map1[i, j] = p_val

In [ ]:
# convert to xarray DataArray
p_value_da = xr.DataArray(
    data=p_value_map,
    coords={'latitude': whiplash_month.latitude, 'longitude': whiplash_month.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

In [ ]:
# convert to xarray DataArray
p_value_da_1 = xr.DataArray(
    data=p_value_map_1,
    coords={'latitude': whiplash_month.latitude, 'longitude': whiplash_month.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

In [ ]:
# convert to xarray DataArray
p_value_da_3 = xr.DataArray(
    data=p_value_map_3,
    coords={'latitude': whiplash_month.latitude, 'longitude': whiplash_month.longitude},
    dims=('latitude', 'longitude'),
    name='p_value',
)

In [ ]:
# Keep only values where p < 0.05 (significant)
trigger_rate_significant_da_5 = trigger_rate_da_5.where(p_value_da < 0.05)

In [ ]:
# Keep only values where p < 0.05 (significant)
trigger_rate_significant_da_1 = trigger_rate_da_1.where(p_value_da_1 < 0.05)

In [ ]:
# Keep only values where p < 0.05 (significant)
trigger_rate_significant_da_3 = trigger_rate_da_3.where(p_value_da_3 < 0.05)

In [ ]:
trigger_rate_significant_da_3.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_1980_2019_3.nc")

In [ ]:
trigger_rate_significant_da_5.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_1980_2019_5.nc")

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from matplotlib.ticker import MultipleLocator, AutoMinorLocator

fig, ax1 = plt.subplots(
    1, 1,
    subplot_kw={'projection': ccrs.PlateCarree(central_longitude=0)},
    figsize=(10, 8),
    constrained_layout=False
)

[x.set_linewidth(1.8) for x in ax1.spines.values()]    
ax1.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

# normalizer (you can change vmin/vmax)
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter = 0.5, vmax=1)

Main_plot = trigger_rate_significant_da_5.plot(
    ax=ax1,
    cmap='turbo',
    norm=norm,
    add_labels=False,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax1.coastlines(resolution='10m', linewidth=1)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', edgecolor='black', linewidth=0.6, zorder=2)
#ax1.add_feature(cfeature.BORDERS, linewidth=0.6)
ax1.set_title(
    r'Trigger Coincidence Rate between Whiplash & Water Stress '
    r'($\Delta T = 5$ months) (1980–2019)',
    fontsize=13,
    pad=8,
    fontname='Times New Roman',
    weight='bold'
)
fig.subplots_adjust(top=0.9, right=0.8)

lat_vals = trigger_rate_significant_da_5['latitude'].values
lon_vals = trigger_rate_significant_da_5['longitude'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
vals = trigger_rate_significant_da_5.values

mask = np.isfinite(vals)

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

sc = ax1.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,                 # SAME norm as raster
    s=3,                      # <-- increase this (try 20–60)
    marker='o',
    edgecolors='black',
    linewidths=0.1,
    transform=ccrs.PlateCarree(),
    zorder=5
)

# --------------------------- # GRID + TICKS (NO Lat/Lon text) # --------------------------- 
gl = ax1.gridlines( draw_labels=False, linewidth=0.6, color='black', linestyle='--', alpha=0.5 )
# Colorbar
cbar_ax = fig.add_axes([0.82, 0.23, 0.02, 0.5])
vmin = 0
vmax = 1  # e.g., 1.21

tick_interval = 0.1
ticks = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]   # last tick slightly below vmax
tick_labels = ["0.0", "0.2", "0.4", "0.6", "0.8", "1.0"]
#tick_labels[-1] = "≥0.6"   # replace last tick label

# ---- MANUAL LABEL FOR EXTENSION ----
cbar.ax.text(
    0.5, 1.02, "≥0.3",
    ha="center", va="bottom",
    transform=cbar.ax.transAxes,
    fontsize=11.7, fontweight="bold"
)
cbar = plt.colorbar(Main_plot, cax=cbar_ax, orientation='vertical', ticks=ticks)
cbar.ax.set_yticklabels(tick_labels)
cbar.outline.set_linewidth(1.2)
cbar.ax.tick_params(which='both', labelsize=11.7, width=1, length=5, direction='in')
cbar.ax.set_ylabel('Trigger coincidence rate', labelpad=1, fontsize=13, fontweight='bold')

print("Number of plotted grid points:", np.sum(mask))

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from matplotlib.ticker import MultipleLocator, AutoMinorLocator

fig, ax1 = plt.subplots(
    1, 1,
    subplot_kw={'projection': ccrs.PlateCarree(central_longitude=0)},
    figsize=(10, 8),
    constrained_layout=False
)

[x.set_linewidth(1.8) for x in ax1.spines.values()]    
ax1.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

# normalizer (you can change vmin/vmax)
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter = 0.5, vmax=1)

Main_plot = trigger_rate_significant_da_1.plot(
    ax=ax1,
    cmap='turbo',
    norm=norm,
    add_labels=False,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

ax1.coastlines(resolution='10m', linewidth=1)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', edgecolor='black', linewidth=0.6, zorder=2)
#ax1.add_feature(cfeature.BORDERS, linewidth=0.6)
ax1.set_title(
    r'Trigger Coincidence Rate between Whiplash & Water Stress '
    r'($\Delta T = 1$ month) (1980–2019)',
    fontsize=13,
    pad=8,
    fontname='Times New Roman',
    weight='bold'
)
fig.subplots_adjust(top=0.9, right=0.8)

lat_vals = trigger_rate_significant_da_1['latitude'].values
lon_vals = trigger_rate_significant_da_1['longitude'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
vals = trigger_rate_significant_da_1.values

mask = np.isfinite(vals)

lon_pts = lon2d[mask]
lat_pts = lat2d[mask]
val_pts = vals[mask]

sc = ax1.scatter(
    lon_pts,
    lat_pts,
    c=val_pts,
    cmap='turbo',
    norm=norm,                 # SAME norm as raster
    s=3,                      # <-- increase this (try 20–60)
    marker='o',
    edgecolors='black',
    linewidths=0.1,
    transform=ccrs.PlateCarree(),
    zorder=5
)

# Colorbar
cbar_ax = fig.add_axes([0.82, 0.23, 0.02, 0.5])
vmin = 0
vmax = 1  # e.g., 1.21

tick_interval = 0.1
ticks = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]   # last tick slightly below vmax
tick_labels = ["0.0", "0.2", "0.4", "0.6", "0.8", "1.0"]
#tick_labels[-1] = "≥0.6"   # replace last tick label

# ---- MANUAL LABEL FOR EXTENSION ----
cbar.ax.text(
    0.5, 1.02, "≥0.3",
    ha="center", va="bottom",
    transform=cbar.ax.transAxes,
    fontsize=11.7, fontweight="bold"
)
cbar = plt.colorbar(Main_plot, cax=cbar_ax, orientation='vertical', ticks=ticks)
cbar.ax.set_yticklabels(tick_labels)
cbar.outline.set_linewidth(1.2)
cbar.ax.tick_params(which='both', labelsize=11.7, width=1, length=5, direction='in')
cbar.ax.set_ylabel('Trigger coincidence rate', labelpad=1, fontsize=13, fontweight='bold')

print("Number of plotted grid points:", np.sum(mask))

In [ ]:
import xarray as xr

In [ ]:
trigger_rate_significant_ds_3 = xr.open_dataset(r"\\10.230.28.136\cez248024\Objective 1\Whiplash results\Cause-Effect\trigger_rate_significant_1980_2019_3.nc")

In [ ]:
trigger_rate_significant_ds_3

In [ ]:
trigger_rate_significant_da_3 = trigger_rate_significant_ds_3['trigger_coincidence_rate']

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ---------------------------
# GLOBAL FONT SETTINGS
# ---------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14

# ---------------------------
# FIGURE
# ---------------------------
fig = plt.figure(figsize=(15, 12))

ax1 = fig.add_axes([0.08, 0.1, 0.70, 0.8],
                   projection=ccrs.PlateCarree())
ax1.set_extent([-180, 180, -90, 90])

# ---------------------------
# NORMALIZATION + COLORMAP
# ---------------------------
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=0.5, vmax=1)
cmap = "viridis"  # color-blind friendly

# ---------------------------
# MAIN PLOT
# ---------------------------
Main_plot = trigger_rate_significant_da_3.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_labels=False,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

# ---------------------------
# COAST + OCEAN
# ---------------------------
ax1.coastlines(resolution='110m', linewidth=1)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=0)

# ---------------------------
# SCATTER OVERLAY
# ---------------------------
lat_vals = trigger_rate_significant_da_3['latitude'].values
lon_vals = trigger_rate_significant_da_3['longitude'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
vals = trigger_rate_significant_da_3.values
mask = np.isfinite(vals)

ax1.scatter(
    lon2d[mask],
    lat2d[mask],
    c=vals[mask],
    cmap=cmap,
    norm=norm,
    s=2,  # bigger for visibility
    marker='o',
    edgecolors='none',
    transform=ccrs.PlateCarree(),
    zorder=5
)

# ---------------------------
# GRIDLINES
# ---------------------------
gl = ax1.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)

# ---------------------------
# LAT/LON TICKS & LABELS
# ---------------------------
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax1.set_xticks(xticks, crs=ccrs.PlateCarree())
ax1.set_yticks(yticks, crs=ccrs.PlateCarree())

ax1.set_xticklabels([f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
                    fontsize=14, fontweight='bold')
ax1.set_yticklabels([f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
                    fontsize=14, fontweight='bold')

ax1.tick_params(axis='both', direction='in', length=6, width=1.5)

# ---------------------------
# BORDER
# ---------------------------
ax1.spines['geo'].set_linewidth(2)
ax1.spines['geo'].set_edgecolor('black')

# ---------------------------
# TITLE
# ---------------------------
ax1.set_title(
    r'Trigger Coincidence Rate between Whiplash & Water Stress (1980–2019)',
    fontsize=16,
    fontweight='bold'
)

# ---------------------------
# COLORBAR (outside plot)

# Shorter colorbar
cbar_ax = fig.add_axes([0.82, 0.25, 0.02, 0.5])  # x, y, width, height
ticks = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
tick_labels = ["0.0", "0.2", "0.4", "0.6", "0.8", "1.0"]

cbar = plt.colorbar(Main_plot, cax=cbar_ax, orientation='vertical', ticks=ticks)
cbar.ax.set_yticklabels(tick_labels, fontsize=14, fontweight='bold', fontname='Times New Roman')
cbar.ax.tick_params(width=1.2, length=5, direction='in')
cbar.set_label('Trigger coincidence rate', fontsize=14, fontweight='bold')

# ---------------------------
# INFO
# ---------------------------
print("Number of plotted grid points:", np.sum(mask))

plt.savefig(
    r"D:\IITD\Doctoral Thesis\Work and Progress ppts\Work\IIT Jammu\Global_Trigger_coincidence_Rate_3.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors

# ---------------------------
# GLOBAL FONT SETTINGS
# ---------------------------
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 14

# ---------------------------
# FIGURE
# ---------------------------
fig = plt.figure(figsize=(10, 8))

ax1 = fig.add_axes([0.08, 0.1, 0.70, 0.8],
                   projection=ccrs.PlateCarree())
ax1.set_extent([-180, 180, -90, 90])

# ---------------------------
# NORMALIZATION + COLORMAP
# ---------------------------
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=0.5, vmax=1)
cmap = "viridis"  # color-blind friendly

# ---------------------------
# MAIN PLOT
# ---------------------------
Main_plot = trigger_rate_significant_da_3.plot(
    ax=ax1,
    cmap=cmap,
    norm=norm,
    add_labels=False,
    add_colorbar=False,
    transform=ccrs.PlateCarree()
)

# ---------------------------
# COAST + OCEAN
# ---------------------------
ax1.coastlines(resolution='110m', linewidth=1)
ax1.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=0)

# ---------------------------
# SCATTER OVERLAY
# ---------------------------
lat_vals = trigger_rate_significant_da_3['latitude'].values
lon_vals = trigger_rate_significant_da_3['longitude'].values

lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)
vals = trigger_rate_significant_da_3.values
mask = np.isfinite(vals)

ax1.scatter(
    lon2d[mask],
    lat2d[mask],
    c=vals[mask],
    cmap=cmap,
    norm=norm,
    s=2,  # bigger for visibility
    marker='o',
    edgecolors='none',
    transform=ccrs.PlateCarree(),
    zorder=5
)

# ---------------------------
# GRIDLINES
# ---------------------------
gl = ax1.gridlines(
    draw_labels=False,
    linewidth=0.6,
    color='black',
    linestyle='--',
    alpha=0.5
)

# ---------------------------
# LAT/LON TICKS & LABELS
# ---------------------------
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)

ax1.set_xticks(xticks, crs=ccrs.PlateCarree())
ax1.set_yticks(yticks, crs=ccrs.PlateCarree())

ax1.set_xticklabels([f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
                    fontsize=14, fontweight='bold')
ax1.set_yticklabels([f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
                    fontsize=14, fontweight='bold')

ax1.tick_params(axis='both', direction='in', length=6, width=1.5)

# ---------------------------
# BORDER
# ---------------------------
ax1.spines['geo'].set_linewidth(2)
ax1.spines['geo'].set_edgecolor('black')

# ---------------------------
# TITLE
# ---------------------------
ax1.set_title(
    r'Trigger Coincidence Rate between Whiplash & Water Stress '
    r'($\Delta T = 3$ month) (1980–2019)',
    fontsize=14,
    fontweight='bold'
)

# ---------------------------
# COLORBAR (outside plot)

# Shorter colorbar
cbar_ax = fig.add_axes([0.82, 0.25, 0.02, 0.5])  # x, y, width, height
ticks = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
tick_labels = ["0.0", "0.2", "0.4", "0.6", "0.8", "1.0"]

cbar = plt.colorbar(Main_plot, cax=cbar_ax, orientation='vertical', ticks=ticks)
cbar.ax.set_yticklabels(tick_labels, fontsize=14, fontweight='bold', fontname='Times New Roman')
cbar.ax.tick_params(width=1.2, length=5, direction='in')
cbar.set_label('Trigger coincidence rate', fontsize=14, fontweight='bold')

# ---------------------------
# INFO
# ---------------------------
print("Number of plotted grid points:", np.sum(mask))

plt.show()

# Hotspots

In [ ]:
import numpy as np

freq_grid = np.zeros((nlat, nlon))
intensity_grid = np.full((nlat, nlon), np.nan)
duration_grid = np.full((nlat, nlon), np.nan)

In [ ]:
for (i, j), ev_list in whiplash_dict_c_e.items():

    if len(ev_list) == 0:
        continue

    # -----------------
    # Frequency
    # -----------------
    freq = len(ev_list)
    avg_freq = freq/40
    freq_grid[i, j] = avg_freq

    # -----------------
    # Intensity
    # -----------------
    intensities = [ev["score"] for ev in ev_list]
    intensity_grid[i, j] = np.mean(intensities)

    # -----------------
    # Duration
    # -----------------
    durations = [
        ev["td_index"] - ev["tc_index"] + 1
        for ev in ev_list
    ]
    duration_grid[i, j] = np.mean(durations)

In [ ]:
import xarray as xr

freq_da = xr.DataArray(
    freq_grid,
    coords={'latitude': lat_values, 'longitude': lon_values},
    dims=('latitude', 'longitude'),
    name='frequency'
)

intensity_da = xr.DataArray(
    intensity_grid,
    coords={'latitude': lat_values, 'longitude': lon_values},
    dims=('latitude', 'longitude'),
    name='mean_intensity'
)

duration_da = xr.DataArray(
    duration_grid,
    coords={'latitude': lat_values, 'longitude': lon_values},
    dims=('latitude', 'longitude'),
    name='mean_duration'
)

In [ ]:
# --- Normalize each metric ---
def normalize_grid(grid):
    mask = ~np.isnan(grid)
    norm_grid = np.full_like(grid, np.nan, dtype=float)
    if np.any(mask):
        g_min, g_max = np.nanmin(grid[mask]), np.nanmax(grid[mask])
        if g_max - g_min > 0:
            norm_grid[mask] = (grid[mask] - g_min) / (g_max - g_min)
        else:
            norm_grid[mask] = 0.0
    return norm_grid

In [ ]:
freq_norm = normalize_grid(freq_grid)
intensity_norm = normalize_grid(intensity_grid)
duration_norm = normalize_grid(duration_grid)

In [ ]:
# --- Compute composite IDF ---
composite_idf = np.nanmean(np.array([freq_norm, intensity_norm, duration_norm]), axis=0)

In [ ]:
composite_idf.max()

In [ ]:
composite_da = xr.DataArray(
    composite_idf,
    coords={"latitude": trigger_rate_significant_da_3.latitude, "longitude": trigger_rate_significant_da_3.longitude},
    dims=["latitude","longitude"],
    name="composite_idf"
)

In [ ]:
hotspot_mask = (composite_da > 0.15) & (trigger_rate_significant_da_3.values > 0.5)

In [ ]:
hotspot_da = xr.DataArray(
    hotspot_mask.astype(int),
    coords={"latitude": trigger_rate_significant_da_3.latitude, "longitude": trigger_rate_significant_da_3.longitude},
    dims=["latitude","longitude"],
    name="hotspot_mask"
)

In [ ]:
hotspot_da.to_netcdf(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\Hotspot_mask_1980_2019.nc")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.lines import Line2D
from matplotlib import rcParams, font_manager

# --- Set default font to Times New Roman ---
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14

# ---------------------------
# FIGURE & AXES
# ---------------------------
fig, ax = plt.subplots(
    1, 1,
    subplot_kw={'projection': ccrs.PlateCarree()},
    figsize=(10, 7)
)

ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())
ax.coastlines(resolution='10m', linewidth=1)
ax.add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=0)

# ---------------------------
# EXTRACT LAT/LON
# ---------------------------
lat_vals = hotspot_da['latitude'].values
lon_vals = hotspot_da['longitude'].values
lon2d, lat2d = np.meshgrid(lon_vals, lat_vals)

# Mask only True values
mask = hotspot_da.values == True
lon_pts = lon2d[mask]
lat_pts = lat2d[mask]

# ---------------------------
# PLOT HOTSPOTS (Color-blind friendly)
# ---------------------------
hotspot_color = '#D55E00'  # Color-blind friendly (reddish-orange)
ax.scatter(
    lon_pts,
    lat_pts,
    color=hotspot_color,
    s=4,  # slightly larger for visibility
    marker='o',
    edgecolors='black',
    linewidths=0.3,
    transform=ccrs.PlateCarree(),
    zorder=5
)

# ---------------------------
# GRIDLINES + LAT/LON TICKS
# ---------------------------

gl = ax.gridlines(
    draw_labels=False,       # hide automatic labels
    linewidth=0.8,
    color='black',
    linestyle='--',
    alpha=0.6,
    zorder=1
)

# Force bold Times New Roman labels
xticks = np.arange(-180, 181, 60)
yticks = np.arange(-90, 91, 30)
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xticklabels([f"{abs(x)}°E" if x >= 0 else f"{abs(x)}°W" for x in xticks],
                   fontsize=14, fontweight='bold', family='Times New Roman')
ax.set_yticklabels([f"{abs(y)}°N" if y >= 0 else f"{abs(y)}°S" for y in yticks],
                   fontsize=14, fontweight='bold', family='Times New Roman')

ax.tick_params(axis='both', direction='in', length=6, width=1.5)

# ---------------------------
# TITLE
# ---------------------------
ax.set_title(
    'Adaptation Relevant Hotspots (1980–2019)',
    fontsize=14,
    fontweight='bold',
    family='Times New Roman'
)

# ---------------------------
# LEGEND
# ---------------------------
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           label='Hotspots',
           markerfacecolor=hotspot_color,
           markeredgecolor='black',
           markersize=8)
]

ax.legend(
    handles=legend_elements,
    loc='lower left',
    frameon=True,
    prop=font_manager.FontProperties(family='Times New Roman', size=14, weight='bold')
)

# ---------------------------
# INFO
# ---------------------------
print("Number of hotspot grid cells:", np.sum(mask))

plt.show()

# Results (Interpretations)

## Distribution of Whiplash Scores

In [ ]:
import xarray as xr

In [ ]:
import json
from collections import defaultdict

with open(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\filtered_whiplash_events.json", "r") as f:
    loaded_dict3 = json.load(f)

In [ ]:
whiplash_dict_c_e = defaultdict(list, {eval(k): v for k, v in loaded_dict3.items()})

In [ ]:
whiplash_dict_c_e

In [ ]:
scores = []

for grid, events in whiplash_dict_c_e.items():
    for event in events:
        scores.append(event['score'])

In [ ]:
scores

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

# --- Global font settings ---
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14
rcParams['font.weight'] = 'bold'

plt.figure(figsize=(8,5))

sns.histplot(scores, bins=30, kde=True)

# Labels and title (explicitly bold to be safe)
plt.xlabel("Whiplash Score", fontsize=14, fontweight='bold')
plt.ylabel("Frequency", fontsize=14, fontweight='bold')
plt.title("Distribution of Whiplash Event Scores", fontsize=14, fontweight='bold')

# Tick formatting
plt.xticks(fontsize=14, fontweight='bold')
plt.yticks(fontsize=14, fontweight='bold')

plt.grid(True, linestyle='--', alpha=0.5)
# ---- Save high resolution ----
plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\distribution_whiplash_scores.tif",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import numpy as np

print("Mean:", np.mean(scores))
print("Median:", np.median(scores))
print("95th percentile:", np.percentile(scores, 95))

In [ ]:
sns.histplot(scores, bins=30, kde=True, stat='probability')

##  Percentage of significant rt values in each climate zone

In [ ]:
trigger_rate_significant_da_3

In [ ]:
trigger_rate_significant_da_3_ds = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_significant_1980_2019_3.nc")

In [ ]:
trigger_rate_significant_da_3 = trigger_rate_significant_da_3_ds['trigger_coincidence_rate']

In [ ]:
trigger_rate_da_3_ds = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Cause-Effect\trigger_rate_all_1980_2019_3.nc")

In [ ]:
trigger_rate_da_3=trigger_rate_da_3_ds['trigger_coincidence_rate']

In [ ]:
trigger_rate_da_3

In [ ]:
import xarray as xr

koppen = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\RobustKGMaps.nc")
print(koppen)

koppen_da = koppen['MasterMap1']
import xarray as xr
import numpy as np

# Extract first letter (major class)
koppen_major = koppen_da.astype(str).str[0]

# Optional: mask invalid values (if any)
valid_classes = ['A', 'B', 'C', 'D', 'E']
#oppen_major = koppen_major.where(np.isin(koppen_major, valid_classes))

import numpy as np

koppen_vals = koppen_da.values

koppen_major = np.full_like(koppen_vals, '', dtype=object)

koppen_major[(koppen_vals >= 1) & (koppen_vals <= 3)] = 'A'
koppen_major[(koppen_vals >= 4) & (koppen_vals <= 7)] = 'B'
koppen_major[(koppen_vals >= 8) & (koppen_vals <= 16)] = 'C'
koppen_major[(koppen_vals >= 17) & (koppen_vals <= 28)] = 'D'
koppen_major[(koppen_vals >= 29) & (koppen_vals <= 30)] = 'E'

for cls in ['A','B','C','D','E']:
    print(cls, np.sum(koppen_major == cls))

rt_all = trigger_rate_da_3.values
rt_sig = trigger_rate_significant_da_3.values

results = []

for cls in ['A','B','C','D','E']:
    
    mask = (koppen_major == cls)
    
    total = np.sum(np.isfinite(rt_all) & mask)
    sig = np.sum(np.isfinite(rt_sig) & mask)
    
    print(cls, total, sig)  # debug
    
    if total > 0:
        sig_pct = (sig / total) * 100
        nonsig_pct = 100 - sig_pct
        
        results.append([cls, sig_pct, nonsig_pct])

import pandas as pd
df = pd.DataFrame(results, columns=['Climate', 'Significant', 'Non-significant'])

label_map = {
    'A': 'Tropical',
    'B': 'Arid',
    'C': 'Temperate',
    'D': 'Cold',
    'E': 'Polar'
}

df['Climate'] = df['Climate'].map(label_map)

import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np

# Font settings
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14
rcParams['font.weight'] = 'bold'

x = np.arange(len(df['Climate']))
width = 0.35

plt.figure(figsize=(9,5))

bars1 = plt.bar(x - width/2, df['Significant'], width, label='Significant')
bars2 = plt.bar(x + width/2, df['Non-significant'], width, label='Non-significant')

# Labels & title
plt.xticks(x, df['Climate'], fontweight='bold')
plt.ylabel("Percentage of Grid Cells (%)", fontweight='bold')
plt.title("Climate-wise Trigger Coincidence Significance", fontweight='bold')

# Value labels (VERY important for PPT clarity)
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height + 1,
                 f"{height:.1f}%", ha='center', fontweight='bold')

plt.legend()
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()

plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\climate_zone_tcr_significance.tiff",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import xarray as xr

koppen = xr.open_dataset(r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\RobustKGMaps.nc")
print(koppen)

In [ ]:
koppen_da = koppen['MasterMap1']

In [ ]:
import xarray as xr
import numpy as np

# Extract first letter (major class)
koppen_major = koppen_da.astype(str).str[0]

# Optional: mask invalid values (if any)
valid_classes = ['A', 'B', 'C', 'D', 'E']
#oppen_major = koppen_major.where(np.isin(koppen_major, valid_classes))

In [ ]:
import numpy as np

koppen_vals = koppen_da.values

koppen_major = np.full_like(koppen_vals, '', dtype=object)

koppen_major[(koppen_vals >= 1) & (koppen_vals <= 3)] = 'A'
koppen_major[(koppen_vals >= 4) & (koppen_vals <= 7)] = 'B'
koppen_major[(koppen_vals >= 8) & (koppen_vals <= 16)] = 'C'
koppen_major[(koppen_vals >= 17) & (koppen_vals <= 28)] = 'D'
koppen_major[(koppen_vals >= 29) & (koppen_vals <= 30)] = 'E'

In [ ]:
print(np.unique(koppen_major))

In [ ]:
for cls in ['A','B','C','D','E']:
    print(cls, np.sum(koppen_major == cls))

In [ ]:
rt_all = trigger_rate_da_3.values
rt_sig = trigger_rate_significant_da_3.values

results = []

for cls in ['A','B','C','D','E']:
    
    mask = (koppen_major == cls)
    
    total = np.sum(np.isfinite(rt_all) & mask)
    sig = np.sum(np.isfinite(rt_sig) & mask)
    
    print(cls, total, sig)  # debug
    
    if total > 0:
        sig_pct = (sig / total) * 100
        nonsig_pct = 100 - sig_pct
        
        results.append([cls, sig_pct, nonsig_pct])

import pandas as pd
df = pd.DataFrame(results, columns=['Climate', 'Significant', 'Non-significant'])

In [ ]:
df

In [ ]:
label_map = {
    'A': 'Tropical',
    'B': 'Arid',
    'C': 'Temperate',
    'D': 'Cold',
    'E': 'Polar'
}

df['Climate'] = df['Climate'].map(label_map)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np

# Font settings
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14
rcParams['font.weight'] = 'bold'

x = np.arange(len(df['Climate']))
width = 0.35

plt.figure(figsize=(9,5))

bars1 = plt.bar(x - width/2, df['Significant'], width, label='Significant')
bars2 = plt.bar(x + width/2, df['Non-significant'], width, label='Non-significant')

# Labels & title
plt.xticks(x, df['Climate'], fontweight='bold')
plt.ylabel("Percentage of Grid Cells (%)", fontweight='bold')
plt.title("Climate-wise Trigger Coincidence Significance", fontweight='bold')

# Value labels (VERY important for PPT clarity)
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, height + 1,
                 f"{height:.1f}%", ha='center', fontweight='bold')

plt.legend()
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()

plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\climate_zone_tcr_significance.tiff",
    dpi=600,
    bbox_inches="tight"
)
plt.show()

In [ ]:
# -----------------------------
# Köppen class labels (30 classes)
# -----------------------------
koppen_labels = {
    1: 'Af',  2: 'Am',  3: 'Aw',
    4: 'BWh', 5: 'BWk', 6: 'BSh', 7: 'BSk',
    8: 'Csa', 9: 'Csb', 10: 'Csc',
    11: 'Cwa', 12: 'Cwb', 13: 'Cwc',
    14: 'Cfa', 15: 'Cfb', 16: 'Cfc',
    17: 'Dsa', 18: 'Dsb', 19: 'Dsc', 20: 'Dsd',
    21: 'Dwa', 22: 'Dwb', 23: 'Dwc', 24: 'Dwd',
    25: 'Dfa', 26: 'Dfb', 27: 'Dfc', 28: 'Dfd',
    29: 'ET', 30: 'EF'
}

In [ ]:
# -----------------------------
# Input trigger coincidence arrays
# -----------------------------
rt_all = trigger_rate_da_3.values
rt_sig = trigger_rate_significant_da_3.values

In [ ]:
# -----------------------------
# Calculate significance percentage
# -----------------------------
results = []

for code, label in koppen_labels.items():
    mask = (koppen_vals == code)

    total = np.sum(np.isfinite(rt_all) & mask)
    sig = np.sum(np.isfinite(rt_sig) & mask)

    if total > 0:
        sig_pct = (sig / total) * 100
        nonsig_pct = 100 - sig_pct

        results.append([code, label, sig_pct, nonsig_pct, total])

In [ ]:
# Create dataframe
df = pd.DataFrame(
    results,
    columns=['Code', 'Climate', 'Significant', 'Non-significant', 'Grid_Count']
)

# Sort by Köppen code
df = df.sort_values('Code').reset_index(drop=True)

print(df)

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams

# -----------------------------
# Plot
# -----------------------------
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14
rcParams['font.weight'] = 'bold'

x = np.arange(len(df))
width = 0.4

plt.figure(figsize=(18, 7))

bars1 = plt.bar(
    x - width/2,
    df['Significant'],
    width,
    label='Significant'
)

bars2 = plt.bar(
    x + width/2,
    df['Non-significant'],
    width,
    label='Non-significant'
)

# Axis labels
plt.xticks(x, df['Climate'], rotation=45, ha='right', fontweight='bold')
plt.ylabel('Percentage of Grid Cells (%)', fontweight='bold')
plt.xlabel('Köppen Climate Class', fontweight='bold')
plt.title('Köppen Climate-wise Trigger Coincidence Significance', fontweight='bold')

# Value labels (optional; may look crowded)
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height >= 5:  # label only sufficiently large bars
            plt.text(
                bar.get_x() + bar.get_width()/2,
                height + 1,
                f'{height:.1f}',
                ha='center',
                va='bottom',
                fontsize=12,
                rotation=90,
                fontweight='bold'
            )

plt.legend()
plt.ylim(0, 110)
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()

plt.savefig(
    r"\\10.230.28.136\cez248024\Preliminary analysis\Whiplash results\Nature_plots\Plots_final\extra figures from compri\koppen_30zone_tcr_significance.tiff",
    dpi=600,
    bbox_inches='tight'
)

plt.show()


## Heatmap of mean TCR and significant percentage

In [ ]:
import numpy as np
import pandas as pd

rt_sig = trigger_rate_significant_da_3.values

results_mean = {}

for cls in ['A','B','C','D','E']:
    
    zone_mask = (koppen_major == cls)
    
    # Extract values in this climate zone
    tcr_vals = rt_sig[zone_mask]
    
    # Mean of valid (non-NaN) values
    mean_val = np.nanmean(tcr_vals)
    
    results_mean[cls] = mean_val

# Convert to DataFrame
df_mean = pd.DataFrame(list(results_mean.items()), columns=['Climate', 'Mean_TCR'])

In [ ]:
df_mean

In [ ]:
import pandas as pd

heatmap_df = pd.DataFrame({
    'Climate': df['Climate'].values,
    'Mean_TCR': df_mean['Mean_TCR'].values,
    'Significant': df['Significant'].values
})

heatmap_df = heatmap_df.set_index('Climate')

In [ ]:
heatmap_df

In [ ]:
heatmap_norm = heatmap_df.copy()

for col in heatmap_norm.columns:
    heatmap_norm[col] = (heatmap_norm[col] - heatmap_norm[col].min()) / \
                        (heatmap_norm[col].max() - heatmap_norm[col].min())

In [ ]:
print(heatmap_df)
print(heatmap_norm)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams

# Font settings
rcParams['font.family'] = 'Times New Roman'
rcParams['font.size'] = 14
rcParams['font.weight'] = 'bold'

plt.figure(figsize=(6,4))

sns.heatmap(
    heatmap_norm,
    annot=heatmap_df,     # show actual values
    fmt=".2f",
    cmap="coolwarm",
    cbar_kws={'label': 'Normalized Value'}
)

plt.title("Climate-wise Mean TCR and Significance", fontweight='bold')

plt.xticks(fontweight='bold')
plt.yticks(fontweight='bold', rotation=0)

plt.tight_layout()
plt.show()

## Percentage of cells with significant coincidence

In [ ]:
import numpy as np

total_cells = np.isfinite(trigger_rate_da_3.values).sum()

In [ ]:
significant_cells = np.isfinite(trigger_rate_significant_da_3.values).sum()

In [ ]:
percentage = (significant_cells / total_cells) * 100

print(f"Global % of significant trigger coincidence: {percentage:.2f}%")

In [ ]:
rt_sig = trigger_rate_significant_da_3.values

rt_sig_positive = rt_sig[(rt_sig > 0) & np.isfinite(rt_sig)]

min_sig_positive = np.min(rt_sig_positive)

print(f"Minimum Significant RT (>0): {min_sig_positive:.3f}")

In [ ]:
np.nanpercentile(rt_sig, 95)

In [ ]:
np.nanpercentile(rt_sig, 25)

In [ ]:
whiplash_score_da

In [ ]:
whiplash_mask = (whiplash_score_da > 0).any(dim='time')

In [ ]:
sig_mask = np.isfinite(trigger_rate_significant_da_3)

In [ ]:
# Total whiplash grid cells
total_whiplash = np.sum(whiplash_mask.values)

# Whiplash + significant trigger coincidence
whiplash_and_sig = np.sum(whiplash_mask.values & sig_mask.values)

In [ ]:
percentage = (whiplash_and_sig / total_whiplash) * 100

print(f"% of whiplash grid cells with significant trigger coincidence: {percentage:.2f}%")